In [4]:
"""
Neumann problem on [-1,1]^d with predetermined ReLU^k features (weak / H1 form).

PDE:  -div(α ∇u) + u = f, with Neumann boundary data g_N.
Solve for outer coefficients by assembling the H1 variational least-squares
system (mass + stiffness + Neumann boundary terms) via
minimize_linear_layer_H1_explicit_assemble_efficient_general_dim.
Inner weights are predetermined / structured (e.g. sphere sampling) with
redundant-neuron removal; derivatives of ReLU^k features are coded explicitly.

Contrast with neumann_problem_PINN.ipynb, which uses a strong-form PINN
residual + Dirichlet collocation with random ELM features.

Changelog:
  - new function: minimize_linear_layer_H1_explicit_assemble_efficient_general_dim
  - 2025 Mar 6th: general dimension d.
  - 2025 Mar 12th: fixed Monte-Carlo domain scaling bug; code working.
"""
import torch
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt
import time
import sys
from scipy.sparse import linalg
from pathlib import Path
import itertools
import sympy as sp
import math  
if torch.cuda.is_available():  
    device = "cuda" 
else:  
    device = "cpu" 

from scipy.stats import norm
from sklearn.preprocessing import normalize
from scipy.stats.qmc import Sobol

torch.set_default_dtype(torch.float64)
pi = torch.tensor(np.pi,dtype=torch.float64)
ZERO = torch.tensor([0.]).to(device)
class model(nn.Module):
    """ ReLU k shallow neural network
    Parameters: 
    input size: input dimension
    hidden_size1 : number of hidden layers 
    num_classes: output classes 
    k: degree of relu functions
    """
    def __init__(self, input_size, hidden_size1, num_classes,k = 1):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size1)
        self.fc2 = nn.Linear(hidden_size1, num_classes,bias = False)
        self.k = k 
    def forward(self, x):
        u1 = self.fc2(F.relu(self.fc1(x))**self.k)
        return u1
    def evaluate_derivative(self, x, i):
        if self.k == 1:
            u1 = self.fc2(torch.heaviside(self.fc1(x),ZERO) * self.fc1.weight.t()[i-1:i,:] )
        else:
            u1 = self.fc2(self.k*F.relu(self.fc1(x))**(self.k-1) *self.fc1.weight.t()[i-1:i,:] )  
        return u1

def plot_2D(f): 
    
    Nx = 400
    Ny = 400 
    xs = np.linspace(-1, 1, Nx)
    ys = np.linspace(-1, 1, Ny)
    x, y = np.meshgrid(xs, ys, indexing='xy')
    xy_comb = np.stack((x.flatten(),y.flatten())).T
    xy_comb = torch.tensor(xy_comb)
    z = f(xy_comb).reshape(Nx,Ny)
    z = z.detach().numpy()
    plt.figure(dpi=200)
    ax = plt.axes(projection='3d')
    ax.plot_surface(x , y , z )

    plt.show()

def plot_subdomains(my_model):
    x_coord =torch.linspace(0,1,200)
    wi = my_model.fc1.weight.data
    bi = my_model.fc1.bias.data 
    for i, bias in enumerate(bi):  
        if wi[i,1] !=0: 
            plt.plot(x_coord, - wi[i,0]/wi[i,1]*x_coord - bias/wi[i,1])
        else: 
            plt.plot(x_coord,  - bias/wi[i,0]*torch.ones(x_coord.size()))

    plt.xlim([0,1])
    plt.ylim([0,1])
    plt.legend()
    plt.show()
    return 0   

## Initialization
def adjust_neuron_position(my_model,target=None):
    counter = 0 
    # positions = torch.tensor([[0.,0.],[0.,1.],[1.,1.],[1.,0.]])
    positions = torch.tensor([[-1.,-1.],[-1.,1.],[1.,1.],[1.,-1.]])
    neuron_num = my_model.fc1.bias.size(0)
    for i in range(neuron_num): 
        w = my_model.fc1.weight.data[i:i+1,:]
        b = my_model.fc1.bias.data[i]
        values = torch.matmul(positions,w.T) # + b
        left_end = - torch.max(values)
        right_end = - torch.min(values) 
        off_set = (right_end - left_end)/1000 
        if b <= left_end + off_set: # nearly vanishing
            b = torch.rand(1)*(right_end - left_end - off_set*2) + left_end + off_set 
            my_model.fc1.bias.data[i] = b 
        if b >= right_end - off_set: # nearly nonvanishing everywhere
            if counter < 3:
                counter += 1
            else: # 3 or more 
                b = torch.rand(1)*(right_end - left_end - off_set*2) + left_end + off_set
                my_model.fc1.bias.data[i] = b 
    return my_model

In [5]:
def PiecewiseGQ2D_weights_points(Nx, order,bl = [-1,-1],ur = [1,1]): 
    """ A slight modification of PiecewiseGQ2D function that only needs the weights and integration points.
    Parameters
    Allows a symmetric square domain (around 0) with lower left corner at bl and upper right corner at ur 
    ----------
    Nx: int 
        number of intervals along the dimension. No Ny, assume Nx = Ny
    order: int 
        order of the Gauss Quadrature
    Returns
    -------
    long_weights: torch.tensor
    integration_points: torch.tensor
    """
#     print("order: ",order )
    x, w = np.polynomial.legendre.leggauss(order)
    gauss_pts = np.array(np.meshgrid(x,x,indexing='ij')).reshape(2,-1).T
    weights =  (w*w[:,None]).ravel()

    gauss_pts =torch.tensor(gauss_pts)
    weights = torch.tensor(weights)

    h = (ur[0]- bl[0])/Nx # 100 intervals 
    long_weights =  torch.tile(weights,(Nx**2,1))
    long_weights = long_weights.reshape(-1,1)
    long_weights = long_weights * h**2 /4 

    integration_points = torch.tile(gauss_pts,(Nx**2,1))
    scale_factor = h/2 
    integration_points = scale_factor * integration_points

    index = np.arange(0,Nx)  
    ordered_pairs = np.array(np.meshgrid(index,index,indexing='ij'))
    ordered_pairs = ordered_pairs.reshape(2,-1).T

    # print(ordered_pairs)
    # print()
    ordered_pairs = torch.tensor(ordered_pairs)
    # print(ordered_pairs.size())
    ordered_pairs = torch.tile(ordered_pairs, (1,order**2)) # number of GQ points
    # print(ordered_pairs)

    ordered_pairs =  ordered_pairs.reshape(-1,2)
    # print(ordered_pairs)
    translation = ordered_pairs*h + (torch.tensor(bl) + h/2) 
    # print(translation)

    integration_points = integration_points + translation 
#     print(integration_points.size())
    # func_values = integrand2_torch(integration_points)
    return long_weights.to(device), integration_points.to(device)


def PiecewiseGQ3D_weights_points(Nx, order,bl = [-1,-1,-1],ur = [1,1,1]): 
    """ A slight modification of PiecewiseGQ2D function that only needs the weights and integration points.
    Parameters
    ----------

    Nx: int 
        number of intervals along the dimension. No Ny, assume Nx = Ny
    order: int 
        order of the Gauss Quadrature

    Returns
    -------
    long_weights: torch.tensor
    integration_points: torch.tensor
    """

    """
    Parameters
    ----------
    target : 
        Target function 
    Nx: int 
        number of intervals along the dimension. No Ny, assume Nx = Ny
    order: int 
        order of the Gauss Quadrature
    """

    # print("order: ",order )
    x, w = np.polynomial.legendre.leggauss(order)
    gauss_pts = np.array(np.meshgrid(x,x,x,indexing='ij')).reshape(3,-1).T
    weight_list = np.array(np.meshgrid(w,w,w,indexing='ij'))
    weights =   (weight_list[0]*weight_list[1]*weight_list[2]).ravel() 

    gauss_pts =torch.tensor(gauss_pts)
    weights = torch.tensor(weights)

    # h = 1/Nx # 100 intervals 
    h = (ur[0]- bl[0])/Nx # 100 intervals 
    long_weights =  torch.tile(weights,(Nx**3,1))
    long_weights = long_weights.reshape(-1,1)
    long_weights = long_weights * h**3 /8 

    integration_points = torch.tile(gauss_pts,(Nx**3,1))
    # print("shape of integration_points", integration_points.size())
    scale_factor = h/2 
    integration_points = scale_factor * integration_points

    # index = np.arange(1,Nx+1)-0.5
    index = np.arange(0,Nx)  
    ordered_pairs = np.array(np.meshgrid(index,index,index,indexing='ij'))
    ordered_pairs = ordered_pairs.reshape(3,-1).T

    # print(ordered_pairs)
    # print()
    ordered_pairs = torch.tensor(ordered_pairs)
    # print(ordered_pairs.size())
    ordered_pairs = torch.tile(ordered_pairs, (1,order**3)) # number of GQ points
    # print(ordered_pairs)

    ordered_pairs =  ordered_pairs.reshape(-1,3)
    # print(ordered_pairs)
    # translation = ordered_pairs*h 
    translation = ordered_pairs*h + (torch.tensor(bl) + h/2) 
    # print(translation)

    integration_points = integration_points + translation 

    return long_weights.to(device), integration_points.to(device)

def MonteCarlo_Sobol_dDim_weights_points(M ,d = 4,bl = -1,ur = 1):
    
    length = ur - bl
    vol = length ** d 
    Sob_integral = torch.quasirandom.SobolEngine(dimension =d, scramble= False, seed=None) 
    integration_points = Sob_integral.draw(M).double() 
    integration_points = integration_points.to(device) * length - length/2 
    weights = torch.ones(M,1).to(device)/M * vol 
    return weights.to(device), integration_points.to(device) 


def minimize_linear_layer_H1_explicit_assemble_efficient_general_dim(model,target, g_N, weights, integration_points, w_bd, pts_bd, activation = 'relu',solver="direct",memory = 2**29 ):
    """ -div alpha grad u(x) + u = f 
    Parameters
    ----------
    model: 
        nn model
    alpha:
        alpha function
    target:
        rhs function f 
    pts_bd:
        integration points on the boundary, embdedded in the domain 
    """ 
    zero = torch.tensor([0.]).to(device)
    start_time = time.time() 
    w = model.fc1.weight.data 
    b = model.fc1.bias.data 
    neuron_num = b.size(0) 
    dim = integration_points.size(1) 
    M = integration_points.size(0)

    total_size = neuron_num * M # memory, number of floating numbers 
    print('total size: {} {} = {}'.format(neuron_num,M,total_size))
    num_batch = total_size//memory + 1 # divide according to memory
    print("num batches: ",num_batch)
    batch_size = M//num_batch
    start_ind = 0
    end_ind = 0 
    jac = torch.zeros(b.size(0),b.size(0)).to(device)
    rhs = torch.zeros(b.size(0),1).to(device)

    for j in range(0,M,batch_size): # batch operation in data points 
        end_ind = j + batch_size
        basis_value_col = F.relu(integration_points[j:end_ind] @ w.t()+ b)**(model.k) 
        weighted_basis_value_col = basis_value_col * weights[j:end_ind] 
        jac += weighted_basis_value_col.t() @ basis_value_col 
        rhs += weighted_basis_value_col.t() @ (target(integration_points[j:end_ind,:])) 

    # Assemble the boundary condition term <g,v>_{\Gamma_N} 
    size_pts_bd = int(pts_bd.size(0)/(2*dim))
    # M_bc = size_pts_bd 
    # total_size = M_bc * neuron_num 
    # num_batch = total_size//memory + 1 
    # batch_size = M_bc//num_batch
    if g_N != None:
        bcs_N = g_N(dim)
        for ii, g_ii in bcs_N:
            weighted_g_N = -g_ii(pts_bd[2*ii*size_pts_bd:(2*ii+1)*size_pts_bd,:])* w_bd[2*ii*size_pts_bd:(2*ii+1)*size_pts_bd,:]
            basis_value_bd_col = F.relu(pts_bd[2*ii*size_pts_bd:(2*ii+1)*size_pts_bd,:] @ w.t()+ b)**(model.k)
            rhs += basis_value_bd_col.t() @ weighted_g_N

            weighted_g_N = g_ii(pts_bd[(2*ii+1)*size_pts_bd:(2*ii+2)*size_pts_bd,:])* w_bd[(2*ii+1)*size_pts_bd:(2*ii+2)*size_pts_bd,:]
            basis_value_bd_col = F.relu(pts_bd[(2*ii+1)*size_pts_bd:(2*ii+2)*size_pts_bd,:] @ w.t()+ b)**(model.k)
            rhs += basis_value_bd_col.t() @ weighted_g_N
            
    # Stiffness matrix term in the jacobian 
    for d in range(dim):
        end_ind = 0 
        if model.k == 1:  
            for j in range(0,M,batch_size): 
                end_ind = j + batch_size 
                basis_value_dxi_col = torch.heaviside(integration_points[j:end_ind] @ w.t()+ b, zero) * w.t()[d:d+1,:]
                weighted_basis_value_dx_col = basis_value_dxi_col * weights[j:end_ind] 
                jac += weighted_basis_value_dx_col.t() @ basis_value_dxi_col 
#             basis_value_dxi_col = torch.heaviside(integration_points @ w.t()+ b, zero) * w.t()[d:d+1,:]
#             weighted_basis_value_dx_col = basis_value_dxi_col * weights * coef_alpha 
#             jac += weighted_basis_value_dx_col.t() @ basis_value_dxi_col 

        else: 
            for j in range(0,M,batch_size):  
                end_ind = j + batch_size 
                basis_value_dxi_col = model.k * F.relu(integration_points[j:end_ind] @ w.t()+ b)**(model.k-1) * w.t()[d:d+1,:]
                weighted_basis_value_dx_col = basis_value_dxi_col * weights[j:end_ind]  
                jac += weighted_basis_value_dx_col.t() @ basis_value_dxi_col 
#             basis_value_dxi_col = model.k * F.relu(integration_points @ w.t()+ b)**(model.k-1) * w.t()[d:d+1,:]
#             weighted_basis_value_dx_col = basis_value_dxi_col * weights * coef_alpha  
#             jac += weighted_basis_value_dx_col.t() @ basis_value_dxi_col 

    print("assembling the mass matrix time taken: ", time.time()-start_time) 

    start_time = time.time()    
    if solver == "cg": 
        sol, exit_code = linalg.cg(np.array(jac.detach().cpu()),np.array(rhs.detach().cpu()),tol=1e-12)
        sol = torch.tensor(sol).view(1,-1)
    elif solver == "direct": 
#         sol = np.linalg.inv( np.array(jac.detach().cpu()) )@np.array(rhs.detach().cpu())
        sol = (torch.linalg.solve( jac.detach(), rhs.detach())).view(1,-1)
    elif solver == "ls":
        sol = (torch.linalg.lstsq(jac.detach().cpu(),rhs.detach().cpu(),driver='gelsd').solution).view(1,-1)
        # sol = (torch.linalg.lstsq(jac.detach(),rhs.detach()).solution).view(1,-1) # gpu/cpu, driver = 'gels', cannot solve singular
    print("solving Ax = b time taken: ", time.time()-start_time)
    return sol 

def minimize_linear_layer_H1_explicit_assemble_efficient(model,target,weights, integration_points,activation = 'relu',solver="direct" ):

    # weights, integration_points = PiecewiseGQ2D_weights_points(Nx, order) 
    # integration_points.requires_grad_(True) 
    start_time = time.time() 
    w = model.fc1.weight.data 
    b = model.fc1.bias.data 
    neuron_num = b.size(0) 

    if activation == 'relu':
        basis_value_col = F.relu(integration_points @ w.t()+ b)**(model.k) 
        if model.k == 1:  
            basis_value_dx_col = torch.heaviside(integration_points @ w.t()+ b, torch.tensor([0.])) * w.t()[0:1,:] 
            basis_value_dy_col = torch.heaviside(integration_points @ w.t()+ b,torch.tensor([0.])) * w.t()[1:2,:] 
        else: 
            basis_value_dx_col = model.k * F.relu(integration_points @ w.t()+ b)**(model.k-1) * w.t()[0:1,:]
            basis_value_dy_col = model.k * F.relu(integration_points @ w.t()+ b)**(model.k-1) * w.t()[1:2,:] 
    # elif activation == 'tanh': 
    #     basis_value_col = torch.tanh(integration_points @ w.t()+ b) 
    #     basis_value_dx_col = tanh_activation_dx(integration_points @ w.t()+ b) * w.t()[0:1,:]
    #     basis_value_dy_col = tanh_activation_dx(integration_points @ w.t()+ b) * w.t()[1:2,:]
    # elif activation == 'gaussian':
    #     basis_value_col = Gaussian_activation(integration_points @ w.t()+ b)
    #     basis_value_dx_col = Gaussian_activation_dx(integration_points @ w.t()+ b) * w.t()[0:1,:]
    #     basis_value_dy_col = Gaussian_activation_dx(integration_points @ w.t()+ b) * w.t()[1:2,:]
    # elif activation == 'cosine':
    #     basis_value_col = cosine_activation(integration_points @ w.t()+ b) 
    #     basis_value_dx_col = cosine_activation_dx(integration_points @ w.t()+ b) * w.t()[0:1,:]
    #     basis_value_dy_col = cosine_activation_dx(integration_points @ w.t()+ b) * w.t()[1:2,:] 

    weighted_basis_value_col = basis_value_col * weights 
    jac1 = weighted_basis_value_col.t() @ basis_value_col  # mass matrix 
    rhs = weighted_basis_value_col.t() @ (target(integration_points)) 
    print("assembling the mass matrix time taken: ", time.time()-start_time) 

    start_time = time.time() 
    weighted_basis_value_dx_col = basis_value_dx_col * weights
    weighted_basis_value_dy_col = basis_value_dy_col * weights
    jac2 = weighted_basis_value_dx_col.t() @ basis_value_dx_col + weighted_basis_value_dy_col.t() @ basis_value_dy_col 
    print("assembling the stiffness matrix time taken: ", time.time()-start_time)   
    jac = jac1 + jac2    
    
    start_time = time.time()    
    if solver == "cg": 
        sol, exit_code = linalg.cg(np.array(jac.detach().cpu()),np.array(rhs.detach().cpu()),tol=1e-12)
        sol = torch.tensor(sol).view(1,-1)
    elif solver == "direct": 
#         sol = np.linalg.inv( np.array(jac.detach().cpu()) )@np.array(rhs.detach().cpu())
        sol = (torch.linalg.solve( jac.detach(), rhs.detach())).view(1,-1)
    elif solver == "ls":
        sol = (torch.linalg.lstsq(jac.detach().cpu(),rhs.detach().cpu(),driver='gelsd').solution).view(1,-1)
        # sol = (torch.linalg.lstsq(jac.detach(),rhs.detach()).solution).view(1,-1) # gpu/cpu, driver = 'gels', cannot solve singular
    print("solving Ax = b time taken: ", time.time()-start_time)
    return sol 


In [6]:
## test quadrature rule 
def integral(x):
    return torch.sin(pi*x[:,0:1])*torch.sin(pi*x[:,1:2])*torch.sin(pi*x[:,2:3])

integration_weights,integration_points = PiecewiseGQ3D_weights_points(20, 3,[-1,-1,-1],[1,1,1])
integral_value = torch.sum(integral(integration_points)*integration_weights) 
print(integral_value) 

def integral2(x):
    return x[:,0:1]**6 * x[:,1:2]**2 * x[:,2:3]**3

integration_weights,integration_points = PiecewiseGQ3D_weights_points(20, 3,[-1,-1,-1],[1,1,1])
integral_value = torch.sum(integral2(integration_points)*integration_weights) 
print(integral_value) 


def integral3(x):
    return x[:,0:1]**6 * x[:,1:2]**2 * x[:,2:3]**4

integration_weights,integration_points = PiecewiseGQ3D_weights_points(50, 3,[-1,-1,-1],[1,1,1])
integral_value = torch.sum(integral3(integration_points)*integration_weights) 
print("exact value: ",8/105)
torch.set_printoptions(precision=13)
print("approximated value:",integral_value) 


tensor(-1.8431e-18)
tensor(3.5779e-17)
exact value:  0.0761904761904762
approximated value: tensor(0.0761904761897)


In [7]:
def show_convergence_order(err_l2,err_h10,exponent,dict_size, filename,write2file = False):
    
    if write2file:
        file_mode = "a" if os.path.exists(filename) else "w"
        f_write = open(filename, file_mode)
    
    neuron_nums = [2**j for j in range(2,exponent+1)]
    err_list = [err_l2[i] for i in neuron_nums ]
    err_list2 = [err_h10[i] for i in neuron_nums ] 
    # f_write.write('M:{}, relu {} \n'.format(M,k))
    if write2file:
        f_write.write('dictionary size: {}\n'.format(dict_size))
        f_write.write("neuron num \t\t error \t\t order \t\t h10 error \\ order \n")
    print("neuron num \t\t error \t\t order")
    for i, item in enumerate(err_list):
        if i == 0: 
            # print(neuron_nums[i], end = "\t\t")
            # print(item, end = "\t\t")
            
            # print("*")
            print("{} \t\t {:.6f} \t\t * \t\t {:.6f} \t\t * \n".format(neuron_nums[i],item, err_list2[i] ) )
            if write2file: 
                f_write.write("{} \t\t {} \t\t * \t\t {} \t\t * \n".format(neuron_nums[i],item, err_list2[i] ))
        else: 
            # print(neuron_nums[i], end = "\t\t")
            # print(item, end = "\t\t") 
            # print(np.log(err_list[i-1]/err_list[i])/np.log(2))
            print("{} \t\t {:.6f} \t\t {:.6f} \t\t {:.6f} \t\t {:.6f} \n".format(neuron_nums[i],item,np.log(err_list[i-1]/err_list[i])/np.log(2),err_list2[i] , np.log(err_list2[i-1]/err_list2[i])/np.log(2) ) )
            if write2file: 
                f_write.write("{} \t\t {} \t\t {} \t\t {} \t\t {} \n".format(neuron_nums[i],item,np.log(err_list[i-1]/err_list[i])/np.log(2),err_list2[i] , np.log(err_list2[i-1]/err_list2[i])/np.log(2) ))
    if write2file:     
        f_write.write("\n")
        f_write.close()

def show_convergence_order_latex(err_l2,err_h10,exponent): 
    neuron_nums = [2**j for j in range(2,exponent+1)]
    err_list = [err_l2[i] for i in neuron_nums ]
    err_list2 = [err_h10[i] for i in neuron_nums ] 
    print("neuron num  & \t $\|u-u_n \|_{L^2}$ & \t order & \t $ | u -u_n |_{H^1}$ & \t order \\\ \hline \hline ")
    for i, item in enumerate(err_list):
        if i == 0: 
            print("{} \t\t & {:.6f} &\t\t * & \t\t {:.6f} & \t\t *  \\\ \hline  \n".format(neuron_nums[i],item, err_list2[i] ) )   
        else: 
            print("{} \t\t &  {:.3e} &  \t\t {:.2f} &  \t\t {:.3e} &  \t\t {:.2f} \\\ \hline  \n".format(neuron_nums[i],item,np.log(err_list[i-1]/err_list[i])/np.log(2),err_list2[i] , np.log(err_list2[i-1]/err_list2[i])/np.log(2) ) )

## helper functions 

# show convergence order 
def output_convergence_order(neuron_nums,err_list_l2, err_list_h2): 
    print("$n$ & \t $\|u-u_n \|_{L^2}$ & \t order  & $ |u-u_n |_{H^1}$ & 	 order\\\ \hline \hline ")
    for i, item in enumerate(err_list_l2):
        if i == 0: 
            print("{} \t\t & {:.3e} &\t\t * & {:.3e} &  *  \\\ \hline  \n".format(neuron_nums[i],item, err_list_h2[i]) )   
        else: 
            print("{} \t\t &  {:.3e} &  \t\t {:.2f}  &  {:.3e} &  {:.2f}  \\\ \hline  \n".format(neuron_nums[i],item, np.log(err_list_l2[i-1]/err_list_l2[i])/np.log(neuron_nums[i]/neuron_nums[i-1]), err_list_h2[i], np.log(err_list_h2[i-1]/err_list_h2[i])/np.log(neuron_nums[i]/neuron_nums[i-1]) ) )


def compute_l2_error(u_exact,my_model,M,batch_size_2,weights,integration_points): 
    err = 0 
    if my_model == None: 
        for jj in range(0,M,batch_size_2): 
            end_index = jj + batch_size_2 
            func_values = u_exact(integration_points[jj:end_index,:])
            err += torch.sum(func_values**2 * weights[jj:end_index,:])
    else: 
        for jj in range(0,M,batch_size_2): 
            end_index = jj + batch_size_2 
            func_values = u_exact(integration_points[jj:end_index,:]) - my_model(integration_points[jj:end_index,:]).detach()
            err += torch.sum(func_values**2 * weights[jj:end_index,:])
    return err**0.5 

def compute_gradient_error(u_exact_grad,my_model,M,batch_size_2,weights,integration_points):
    """
    Parameters
    ----------
    u_exact_grad: list or None
        a list that contains ways of evaluating partial derivatives that gives the gradient  
    """
    err_h10 = 0 
     # initial gradient error 
    if u_exact_grad != None and my_model!=None:
        u_grad = u_exact_grad() 
        for ii, grad_i in enumerate(u_grad): 
            for jj in range(0,M,batch_size_2): 
                end_index = jj + batch_size_2 
                my_model_dxi = my_model.evaluate_derivative(integration_points[jj:end_index,:],ii+1).detach() 
                err_h10 += torch.sum((grad_i(integration_points[jj:end_index,:]) - my_model_dxi)**2 * weights[jj:end_index,:])
    elif u_exact_grad != None and my_model==None:
        u_grad = u_exact_grad() 
        for grad_i in u_grad: 
            for jj in range(0,M,batch_size_2): 
                end_index = jj + batch_size_2 
                err_h10 += torch.sum((grad_i(integration_points[jj:end_index,:]))**2 * weights[jj:end_index,:])
    return err_h10**0.5

## RFM or pre-determined feature  

In [11]:
def initialize_model_1(my_model):
    # w ~ U(S^1), b ~ U(-1.42,1.42) 
    neuron_nums = my_model.fc1.bias.size(0)
    samples = torch.rand(neuron_nums,2) 
    T =torch.tensor([[2*pi,0],[0,2.84]])
    shift = torch.tensor([0,-1.42]) 
    samples = samples@T + shift 
    theta = samples[:,0].reshape(neuron_nums,1)
    W1 = torch.cos(theta)
    W2 = torch.sin(theta)
    W = torch.cat((W1,W2),1) # N1 x 2
    b = samples[:,1].reshape(neuron_nums,1)
    my_model.fc1.weight.data[:,:] = W[:,:]
    my_model.fc1.bias.data[:] = b[:,0] 
    
    return my_model 

def initialize_model_2(my_model,dim = 2):
    # (w,b) ~ U(S^2)
    neuron_nums = my_model.fc1.bias.size(0)
    points = torch.randn(neuron_nums,dim + 1)
    points = points/torch.norm(points, dim=1, keepdim=True)
    my_model.fc1.weight.data[:,:] = points[:,0:dim]
    my_model.fc1.bias.data[:] = points[:,dim]  
    return my_model 


def initialize_model_2_qmc(my_model,dim = 2):
    # (w,b) ~ U(S^2)
    neuron_nums = my_model.fc1.bias.size(0)
    sobol_engine = Sobol(dim+1,scramble=False)
    u = sobol_engine.random(n=neuron_nums)
    
    # points = torch.randn(neuron_nums,dim + 1)
    epsilon = np.finfo(float).eps
    u = np.clip(u, epsilon, 1 - epsilon)

    # Inverse CDF to get standard normal points
    z = norm.ppf(u)

    # Normalize to project onto sphere
    x = normalize(z, axis=1)
    # Convert to torch tensor
    points = torch.tensor(x, dtype=torch.float64)
    points = points/torch.norm(points, dim=1, keepdim=True)
    my_model.fc1.weight.data[:,:] = points[:,0:dim]
    my_model.fc1.bias.data[:] = points[:,dim]  
    return my_model 

def initialize_model_3(my_model):
    # generate a uniform grid on S^2 
    neuron_nums = my_model.fc1.bias.size(0) 

    indices = torch.arange(0, neuron_nums, dtype=torch.float) + 0.5
    phi = torch.acos(1 - 2*indices/neuron_nums)
    theta = pi * (1 + 5**0.5) * indices
    x = torch.sin(phi) * torch.cos(theta)
    y = torch.sin(phi) * torch.sin(theta)
    z = torch.cos(phi)

    points = torch.stack((x, y, z), dim=1)
    my_model.fc1.weight.data[:,:] = points[:,0:2]
    my_model.fc1.bias.data[:] = points[:,2]
    return my_model 

def remove_redundant_neuron(my_model, dims = 3, choice = 2): 
    ##  choice 1:  [0,1]^d, choice 2: [-1,1]^d
    def create_mesh_grid(dims, pts):
        mesh = torch.tensor(list(itertools.product(pts,repeat=dims)))
        vertices = mesh.reshape(len(pts) ** dims, -1) 
        return vertices
    counter = 0 
    # positions = torch.tensor([[0.,0.],[0.,1.],[1.,1.],[1.,0.]])
    # pts = torch.tensor([0.,1.]) # for domain [0,1]^d 
    if choice == 1: 
        pts = torch.tensor([0.,1.])
    elif choice == 2:
        pts = torch.tensor([-1.,1.])# for domain [-1,1]^d 
    elif choice == 3:
        pts = torch.tensor([-1./2,1./2])# for domain [-1,1]^d 
    positions = create_mesh_grid(dims,pts) 
    neuron_num = my_model.fc1.bias.size(0)
    relu_k = my_model.k 
    recorded_neurons = []
    poly_dofs = math.comb(relu_k + dims, dims)
    for i in range(neuron_num): 
        w = my_model.fc1.weight.data[i:i+1,:]
        b = my_model.fc1.bias.data[i]
        values = torch.matmul(positions,w.T)
        left_end = - torch.max(values)
        right_end = - torch.min(values)
        offset = (right_end - left_end)/50
        if b > left_end + offset/2 and b < right_end - offset/2: 
            recorded_neurons.append((w, b))
        elif b >= right_end - offset/2 and counter < poly_dofs:
            recorded_neurons.append((w, b))
            counter += 1

    new_neuron_num = len(recorded_neurons)
    new_model = model(dims, new_neuron_num, 1, k=relu_k).to(device)
    for i, (w, b) in enumerate(recorded_neurons):
        new_model.fc1.weight.data[i:i+1,:] = w
        new_model.fc1.bias.data[i] = b
    print("Number of neurons removed: ", neuron_num - new_neuron_num)
    print("Number of neurons left: ", new_neuron_num)  
    return new_model


## 2D example 

In [12]:
##1. exact solution: sin(pi/2 x)sin(pi/2 y) 
def u_exact(x): 
    z = torch.sin(pi/2*x[:,0:1])*torch.sin(pi/2*x[:,1:2] ) # * torch.sin(pi/2*x[:,2:3])  
    return z 
def u_x(x):
    z = (pi/2)*torch.cos(pi/2*x[:,0:1])*torch.sin(pi/2*x[:,1:2] ) # * torch.sin(pi/2*x[:,2:3])  
    return z
def u_y(x):
    z = (pi/2)*torch.sin(pi/2*x[:,0:1])*torch.cos(pi/2*x[:,1:2] ) # * torch.sin(pi/2*x[:,2:3])  
    return z 
def target(x):
    z = (2 * (pi/2)**2 + 1)*torch.sin(pi/2*x[:,0:1])*torch.sin(pi/2*x[:,1:2] )   
    return z

# ## 2. exact solution (x**2 - 1)(y**2 -1)

# x_sym, y_sym = sp.symbols('x y')
# u_expr = ((x_sym**2 - 1))**2 * ((y_sym**2 - 1))**2

# # Compute symbolic derivatives (e.g., first derivative with respect to x)
# u_x_expr = sp.diff(u_expr, x_sym)
# u_y_expr = sp.diff(u_expr, y_sym)
# # Compute higher-order derivatives if needed:
# u_xx_expr = sp.diff(u_expr, x_sym, 2)
# u_yy_expr = sp.diff(u_expr, y_sym, 2)
# u_xy_expr = sp.diff(u_expr, x_sym, y_sym)
 
# u_exact_sym_func = sp.lambdify((x_sym, y_sym), u_expr, modules='numpy')
# u_x_sym_func     = sp.lambdify((x_sym, y_sym), u_x_expr, modules='numpy')
# u_y_sym_func     = sp.lambdify((x_sym, y_sym), u_y_expr, modules='numpy')
# u_xx_sym_func    = sp.lambdify((x_sym, y_sym), u_xx_expr, modules='numpy')
# u_yy_sym_func    = sp.lambdify((x_sym, y_sym), u_yy_expr, modules='numpy')
# u_xy_sym_func    = sp.lambdify((x_sym, y_sym), u_xy_expr, modules='numpy')

# # Define wrapper functions that accept PyTorch tensors as input and return torch tensors.
# def u_exact(x_tensor):
#     # Assume x_tensor is a tensor of shape (N, 2) where each row is (x, y)
#     x_np = x_tensor.detach().cpu().numpy()
#     # Evaluate the symbolic function using the first and second columns
#     result_np = u_exact_sym_func(x_np[:, 0], x_np[:, 1]).reshape(-1,1)
#     # Convert result to a torch tensor, preserving the device and dtype of the input
#     return torch.from_numpy(np.array(result_np)).to(x_tensor.device).type(x_tensor.dtype)

# def u_x(x_tensor):
#     x_np = x_tensor.detach().cpu().numpy()
#     result_np = u_x_sym_func(x_np[:, 0], x_np[:, 1]).reshape(-1,1)
#     return torch.from_numpy(np.array(result_np)).to(x_tensor.device).type(x_tensor.dtype)

# def u_y(x_tensor):
#     x_np = x_tensor.detach().cpu().numpy()
#     result_np = u_y_sym_func(x_np[:, 0], x_np[:, 1]).reshape(-1,1)
#     return torch.from_numpy(np.array(result_np)).to(x_tensor.device).type(x_tensor.dtype)

# def u_xx(x_tensor):
#     x_np = x_tensor.detach().cpu().numpy()
#     result_np = u_xx_sym_func(x_np[:, 0], x_np[:, 1]).reshape(-1,1)
#     return torch.from_numpy(np.array(result_np)).to(x_tensor.device).type(x_tensor.dtype)

# def u_yy(x_tensor):
#     x_np = x_tensor.detach().cpu().numpy()
#     result_np = u_yy_sym_func(x_np[:, 0], x_np[:, 1]).reshape(-1,1)
#     return torch.from_numpy(np.array(result_np)).to(x_tensor.device).type(x_tensor.dtype)

# def target(x_tensor):
#     return - (u_xx(x_tensor) + u_yy(x_tensor) ) + u_exact(x_tensor) 

Nx = 200   
order = 3 
integration_weights, integration_points = PiecewiseGQ2D_weights_points(Nx, order,[-1,-1],[1,1])

neuron_num_list = [25,50,100,200,400,800,1600]
actual_neuron_list = []
err_list_l2 = []
err_list_h1 = []
relu_k = 3 
for neuron_num in neuron_num_list: 
    my_model = model(2,neuron_num,1,relu_k).to(device)
    my_model = initialize_model_3(my_model).to(device)
    my_model = remove_redundant_neuron(my_model.cpu(), dims = 2, choice = 2).to(device) 
    solver = 'direct'
    # sol = minimize_linear_layer_H1_explicit_assemble_efficient(my_model,target,integration_weights, integration_points,activation = 'relu',solver = solver)
    sol = minimize_linear_layer_H1_explicit_assemble_efficient_general_dim(my_model,target,None,integration_weights, integration_points,torch.tensor([]),torch.tensor([]),activation = 'relu',solver = solver)
    my_model.fc2.weight.data[0,:] = sol[:] 
    actual_neuron_list.append(my_model.fc1.bias.size(0)) 
    # compute the error 
    errl2 = (integration_weights.t()@(u_exact(integration_points) - my_model(integration_points).detach())**2)**0.5
    errh1 = integration_weights.t()@(u_x(integration_points) - my_model.evaluate_derivative(integration_points,1).detach())**2
    errh1 += integration_weights.t()@(u_y(integration_points) - my_model.evaluate_derivative(integration_points,2).detach())**2
    errh1 = errh1**0.5 

    print("L2 error: ",errl2)
    print("H1 error: ",errh1) 
    err_list_l2.append(errl2.item())
    err_list_h1.append(errh1.item())
    # plot_2D(my_model.cpu())  

Number of neurons removed:  2
Number of neurons left:  23
total size: 23 360000 = 8280000
num batches:  1
assembling the mass matrix time taken:  0.2219228744506836
solving Ax = b time taken:  0.00018024444580078125
L2 error:  tensor([[0.0104621387220]])
H1 error:  tensor([[0.0831419901464]])
Number of neurons removed:  6
Number of neurons left:  44
total size: 44 360000 = 15840000
num batches:  1
assembling the mass matrix time taken:  0.3443470001220703
solving Ax = b time taken:  0.00026297569274902344
L2 error:  tensor([[0.0016551344853]])
H1 error:  tensor([[0.0195748839582]])
Number of neurons removed:  12
Number of neurons left:  88
total size: 88 360000 = 31680000
num batches:  1
assembling the mass matrix time taken:  0.7484729290008545
solving Ax = b time taken:  0.00029468536376953125
L2 error:  tensor([[0.0005530644220]])
H1 error:  tensor([[0.0082919909392]])
Number of neurons removed:  35
Number of neurons left:  165
total size: 165 360000 = 59400000
num batches:  1
assem

In [13]:
d = 2 
print("L2 order:",0.5 + (2*relu_k +1)/(2*d))
print("H1 order:",0.5 + (2*(relu_k-1) +1)/(2*d))

output_convergence_order(actual_neuron_list,err_list_l2, err_list_h1)

L2 order: 2.25
H1 order: 1.75
$n$ & 	 $\|u-u_n \|_{L^2}$ & 	 order  & $ |u-u_n |_{H^1}$ & 	 order\\ \hline \hline 
23 		 & 1.046e-02 &		 * & 8.314e-02 &  *  \\ \hline  

44 		 &  1.655e-03 &  		 2.84  &  1.957e-02 &  2.23  \\ \hline  

88 		 &  5.531e-04 &  		 1.58  &  8.292e-03 &  1.24  \\ \hline  

165 		 &  8.224e-05 &  		 3.03  &  1.905e-03 &  2.34  \\ \hline  

318 		 &  2.061e-05 &  		 2.11  &  6.049e-04 &  1.75  \\ \hline  

631 		 &  4.219e-06 &  		 2.31  &  1.844e-04 &  1.73  \\ \hline  

1256 		 &  1.248e-06 &  		 1.77  &  7.730e-05 &  1.26  \\ \hline  



## 3D example 

- $ u(x) = \Pi_{i=1}^3 \sin(\frac{\pi}{2} x_i)$ 

In [9]:
##1. exact solution: sin(pi/2 x)sin(pi/2 y) 
def u_exact(x): 
    z = torch.sin(pi/2*x[:,0:1])*torch.sin(pi/2*x[:,1:2] ) * torch.sin(pi/2*x[:,2:3])  
    return z 
def u_x(x):
    z = (pi/2)*torch.cos(pi/2*x[:,0:1])*torch.sin(pi/2*x[:,1:2] ) * torch.sin(pi/2*x[:,2:3])  
    return z
def u_y(x):
    z = (pi/2)*torch.sin(pi/2*x[:,0:1])*torch.cos(pi/2*x[:,1:2] ) * torch.sin(pi/2*x[:,2:3])  
    return z 
def u_z(x):
    z = (pi/2)*torch.sin(pi/2*x[:,0:1])*torch.sin(pi/2*x[:,1:2] ) * torch.cos(pi/2*x[:,2:3])  
    return z 
def target(x):
    z = (3 * (pi/2)**2 + 1)*torch.sin(pi/2*x[:,0:1])*torch.sin(pi/2*x[:,1:2] ) * torch.sin(pi/2*x[:,2:3]) 
    return z
def u_exact_grad(): 
    return [u_x, u_y, u_z]

Nx = 100    
order = 3 
integration_weights, integration_points = PiecewiseGQ3D_weights_points(Nx, order,[-1,-1,-1],[1,1,1])
M = integration_points.size(0)
neuron_num_list = [25,50,100,200,400,800,1600]
actual_neuron_list = []
err_list_l2 = []
err_list_h1 = []
relu_k = 3 
for neuron_num in neuron_num_list: 
    my_model = model(3,neuron_num,1,relu_k).to(device)
    my_model = initialize_model_2(my_model,dim=3).to(device)
    my_model = remove_redundant_neuron(my_model.cpu(), dims = 3, choice = 2).to(device) 
    solver = 'direct'
    # sol = minimize_linear_layer_H1_explicit_assemble_efficient(my_model,target,integration_weights, integration_points,activation = 'relu',solver = solver)
    sol = minimize_linear_layer_H1_explicit_assemble_efficient_general_dim(my_model,target,None,integration_weights, integration_points,torch.tensor([]),torch.tensor([]),activation = 'relu',solver = solver)
    my_model.fc2.weight.data[0,:] = sol[:] 
    actual_neuron_list.append(my_model.fc1.bias.size(0)) 
    # compute the error 
    memory = 2**28 
    num_neuron = 0 if my_model == None else int(my_model.fc1.bias.detach().data.size(0))
    total_size2 = M*(num_neuron+1)
    num_batch2 = total_size2//memory + 1 
    batch_size_2 = M//num_batch2 # in
    
#     errl2 = (integration_weights.t()@(u_exact(integration_points) - my_model(integration_points).detach())**2)**0.5
    errl2 = compute_l2_error(u_exact,my_model,M,batch_size_2,integration_weights,integration_points)
#     errh1 = integration_weights.t()@(u_x(integration_points) - my_model.evaluate_derivative(integration_points,1).detach())**2
#     errh1 += integration_weights.t()@(u_y(integration_points) - my_model.evaluate_derivative(integration_points,2).detach())**2
#     errh1 += integration_weights.t()@(u_z(integration_points) - my_model.evaluate_derivative(integration_points,3).detach())**2 
#     errh1 = errh1**0.5    
    errh1 = compute_gradient_error(u_exact_grad,my_model,M,batch_size_2,integration_weights,integration_points)

    print("L2 error: ",errl2)
    print("H1 error: ",errh1) 
    err_list_l2.append(errl2.item())
    err_list_h1.append(errh1.item())
    # plot_2D(my_model.cpu())  

Number of neurons removed:  1
Number of neurons left:  24
total size: 24 27000000 = 648000000
num batches:  2
assembling the mass matrix time taken:  0.002759695053100586
solving Ax = b time taken:  0.2007002830505371
L2 error:  tensor(0.5061340934719, device='cuda:0')
H1 error:  tensor(1.5927391809939, device='cuda:0')
Number of neurons removed:  2
Number of neurons left:  48
total size: 48 27000000 = 1296000000
num batches:  3
assembling the mass matrix time taken:  0.0030994415283203125
solving Ax = b time taken:  0.3589944839477539
L2 error:  tensor(0.1642518956885, device='cuda:0')
H1 error:  tensor(0.8646167051385, device='cuda:0')
Number of neurons removed:  7
Number of neurons left:  93
total size: 93 27000000 = 2511000000
num batches:  5
assembling the mass matrix time taken:  0.004561424255371094
solving Ax = b time taken:  0.8466994762420654
L2 error:  tensor(0.0339585465902, device='cuda:0')
H1 error:  tensor(0.2515902909596, device='cuda:0')
Number of neurons removed:  11


In [ ]:
d = 3 
print("L2 order:",0.5 + (2*relu_k +1)/(2*d))
print("H1 order:",0.5 + (2*(relu_k-1) +1)/(2*d))

output_convergence_order(actual_neuron_list,err_list_l2, err_list_h1)



L2 order: 1.6666666666666667
H1 order: 1.3333333333333335
$n$ & 	 $\|u-u_n \|_{L^2}$ & 	 order  & $ |u-u_n |_{H^1}$ & 	 order\\ \hline \hline 
24 		 & 5.061e-01 &		 * & 1.593e+00 &  *  \\ \hline  

48 		 &  1.643e-01 &  		 1.62  &  8.646e-01 &  0.88  \\ \hline  

93 		 &  3.396e-02 &  		 2.38  &  2.516e-01 &  1.87  \\ \hline  

189 		 &  1.000e-02 &  		 1.72  &  9.477e-02 &  1.38  \\ \hline  

384 		 &  2.105e-03 &  		 2.20  &  2.689e-02 &  1.78  \\ \hline  

749 		 &  5.761e-04 &  		 1.94  &  9.732e-03 &  1.52  \\ \hline  

1495 		 &  1.887e-04 &  		 1.61  &  3.919e-03 &  1.32  \\ \hline  



## QMC test 


In [ ]:
##1. exact solution: sin(pi/2 x)sin(pi/2 y) 
def u_exact(x): 
    z = torch.sin(pi/2*x[:,0:1])*torch.sin(pi/2*x[:,1:2] ) * torch.sin(pi/2*x[:,2:3])  
    return z 
def u_x(x):
    z = (pi/2)*torch.cos(pi/2*x[:,0:1])*torch.sin(pi/2*x[:,1:2] ) * torch.sin(pi/2*x[:,2:3])  
    return z
def u_y(x):
    z = (pi/2)*torch.sin(pi/2*x[:,0:1])*torch.cos(pi/2*x[:,1:2] ) * torch.sin(pi/2*x[:,2:3])  
    return z 
def u_z(x):
    z = (pi/2)*torch.sin(pi/2*x[:,0:1])*torch.sin(pi/2*x[:,1:2] ) * torch.cos(pi/2*x[:,2:3])  
    return z 
def target(x):
    z = (3 * (pi/2)**2 + 1)*torch.sin(pi/2*x[:,0:1])*torch.sin(pi/2*x[:,1:2] ) * torch.sin(pi/2*x[:,2:3]) 
    return z
def u_exact_grad(): 
    return [u_x, u_y, u_z]

Nx = 25   
order = 3 
integration_weights, integration_points = PiecewiseGQ3D_weights_points(Nx, order,[-1,-1,-1],[1,1,1])
M = integration_points.size(0)
neuron_num_list = [25,50,100,200,400,800]
actual_neuron_list = []
err_list_l2 = []
err_list_h1 = []
relu_k = 3 
for neuron_num in neuron_num_list: 
    my_model = model(3,neuron_num,1,relu_k).to(device)
    my_model = initialize_model_2_qmc(my_model,dim=3).to(device)
    my_model = remove_redundant_neuron(my_model.cpu(), dims = 3, choice = 2).to(device) 
    solver = 'direct'
    # sol = minimize_linear_layer_H1_explicit_assemble_efficient(my_model,target,integration_weights, integration_points,activation = 'relu',solver = solver)
    sol = minimize_linear_layer_H1_explicit_assemble_efficient_general_dim(my_model,target,None,integration_weights, integration_points,torch.tensor([]),torch.tensor([]),activation = 'relu',solver = solver)
    my_model.fc2.weight.data[0,:] = sol[:] 
    actual_neuron_list.append(my_model.fc1.bias.size(0)) 
    # compute the error 
    memory = 2**28 
    num_neuron = 0 if my_model == None else int(my_model.fc1.bias.detach().data.size(0))
    total_size2 = M*(num_neuron+1)
    num_batch2 = total_size2//memory + 1 
    batch_size_2 = M//num_batch2 # in
    
#     errl2 = (integration_weights.t()@(u_exact(integration_points) - my_model(integration_points).detach())**2)**0.5
    errl2 = compute_l2_error(u_exact,my_model,M,batch_size_2,integration_weights,integration_points)
#     errh1 = integration_weights.t()@(u_x(integration_points) - my_model.evaluate_derivative(integration_points,1).detach())**2
#     errh1 += integration_weights.t()@(u_y(integration_points) - my_model.evaluate_derivative(integration_points,2).detach())**2
#     errh1 += integration_weights.t()@(u_z(integration_points) - my_model.evaluate_derivative(integration_points,3).detach())**2 
#     errh1 = errh1**0.5    
    errh1 = compute_gradient_error(u_exact_grad,my_model,M,batch_size_2,integration_weights,integration_points)

    print("L2 error: ",errl2)
    print("H1 error: ",errh1) 
    err_list_l2.append(errl2.item())
    err_list_h1.append(errh1.item())
    # plot_2D(my_model.cpu())  

/var/folders/45/p1_q2drx75983lk7ph_1gx6mhpr5_w/T/ipykernel_10690/737876908.py:32: UserWarning: The balance properties of Sobol' points require n to be a power of 2.
  u = sobol_engine.random(n=neuron_nums)


Number of neurons removed:  2
Number of neurons left:  23
total size: 23 421875 = 9703125
num batches:  1
assembling the mass matrix time taken:  0.3265259265899658
solving Ax = b time taken:  0.016253948211669922
L2 error:  tensor(0.4217183719724)
H1 error:  tensor(1.5501656938890)
Number of neurons removed:  3
Number of neurons left:  47
total size: 47 421875 = 19828125
num batches:  1


/var/folders/45/p1_q2drx75983lk7ph_1gx6mhpr5_w/T/ipykernel_10690/737876908.py:32: UserWarning: The balance properties of Sobol' points require n to be a power of 2.
  u = sobol_engine.random(n=neuron_nums)


assembling the mass matrix time taken:  1.5197010040283203
solving Ax = b time taken:  0.0005180835723876953
L2 error:  tensor(0.1196132451412)
H1 error:  tensor(0.6404140402642)
Number of neurons removed:  4
Number of neurons left:  96
total size: 96 421875 = 40500000
num batches:  1


/var/folders/45/p1_q2drx75983lk7ph_1gx6mhpr5_w/T/ipykernel_10690/737876908.py:32: UserWarning: The balance properties of Sobol' points require n to be a power of 2.
  u = sobol_engine.random(n=neuron_nums)


...

assembling the mass matrix time taken:  11.81216812133789
solving Ax = b time taken:  0.004190206527709961
L2 error:  tensor(0.0017060329480)
H1 error:  tensor(0.0222856212062)
Number of neurons removed:  50
Number of neurons left:  750
total size: 750 421875 = 316406250
num batches:  1


/var/folders/45/p1_q2drx75983lk7ph_1gx6mhpr5_w/T/ipykernel_10690/737876908.py:32: UserWarning: The balance properties of Sobol' points require n to be a power of 2.
  u = sobol_engine.random(n=neuron_nums)


assembling the mass matrix time taken:  32.260366916656494
solving Ax = b time taken:  0.013576030731201172
L2 error:  tensor(0.0004612129206)
H1 error:  tensor(0.0080201838552)
Number of neurons removed:  117
Number of neurons left:  1483
total size: 1483 421875 = 625640625
num batches:  2


/var/folders/45/p1_q2drx75983lk7ph_1gx6mhpr5_w/T/ipykernel_10690/737876908.py:32: UserWarning: The balance properties of Sobol' points require n to be a power of 2.
  u = sobol_engine.random(n=neuron_nums)


assembling the mass matrix time taken:  74.05207395553589
solving Ax = b time taken:  0.1441209316253662
L2 error:  tensor(0.0001607858278)
H1 error:  tensor(0.0035852886078)


In [ ]:
d = 3 
print("L2 order:",0.5 + (2*relu_k +1)/(2*d))
print("H1 order:",0.5 + (2*(relu_k-1) +1)/(2*d))

output_convergence_order(actual_neuron_list,err_list_l2, err_list_h1)



L2 order: 1.6666666666666667
H1 order: 1.3333333333333335
$n$ & 	 $\|u-u_n \|_{L^2}$ & 	 order  & $ |u-u_n |_{H^1}$ & 	 order\\ \hline \hline 
23 		 & 4.217e-01 &		 * & 1.550e+00 &  *  \\ \hline  

47 		 &  1.196e-01 &  		 1.76  &  6.404e-01 &  1.24  \\ \hline  

96 		 &  2.883e-02 &  		 1.99  &  2.212e-01 &  1.49  \\ \hline  

191 		 &  7.638e-03 &  		 1.93  &  7.680e-02 &  1.54  \\ \hline  

385 		 &  1.706e-03 &  		 2.14  &  2.229e-02 &  1.77  \\ \hline  

750 		 &  4.612e-04 &  		 1.96  &  8.020e-03 &  1.53  \\ \hline  

1483 		 &  1.608e-04 &  		 1.55  &  3.585e-03 &  1.18  \\ \hline  



In [15]:
## MC

##1. exact solution: sin(pi/2 x)sin(pi/2 y) 
def u_exact(x): 
    z = torch.sin(pi/2*x[:,0:1])*torch.sin(pi/2*x[:,1:2] ) * torch.sin(pi/2*x[:,2:3])  
    return z 
def u_x(x):
    z = (pi/2)*torch.cos(pi/2*x[:,0:1])*torch.sin(pi/2*x[:,1:2] ) * torch.sin(pi/2*x[:,2:3])  
    return z
def u_y(x):
    z = (pi/2)*torch.sin(pi/2*x[:,0:1])*torch.cos(pi/2*x[:,1:2] ) * torch.sin(pi/2*x[:,2:3])  
    return z 
def u_z(x):
    z = (pi/2)*torch.sin(pi/2*x[:,0:1])*torch.sin(pi/2*x[:,1:2] ) * torch.cos(pi/2*x[:,2:3])  
    return z 
def target(x):
    z = (3 * (pi/2)**2 + 1)*torch.sin(pi/2*x[:,0:1])*torch.sin(pi/2*x[:,1:2] ) * torch.sin(pi/2*x[:,2:3]) 
    return z
def u_exact_grad(): 
    return [u_x, u_y, u_z]

Nx = 25   
order = 3 
integration_weights, integration_points = PiecewiseGQ3D_weights_points(Nx, order,[-1,-1,-1],[1,1,1])
M = integration_points.size(0)
neuron_num_list = [25,50,100,200,400,800]
actual_neuron_list = []
err_list_l2 = []
err_list_h1 = []
relu_k = 3 
for neuron_num in neuron_num_list: 
    my_model = model(3,neuron_num,1,relu_k).to(device)
    my_model = initialize_model_2(my_model,dim=3).to(device)
    my_model = remove_redundant_neuron(my_model.cpu(), dims = 3, choice = 2).to(device) 
    solver = 'direct'
    # sol = minimize_linear_layer_H1_explicit_assemble_efficient(my_model,target,integration_weights, integration_points,activation = 'relu',solver = solver)
    sol = minimize_linear_layer_H1_explicit_assemble_efficient_general_dim(my_model,target,None,integration_weights, integration_points,torch.tensor([]),torch.tensor([]),activation = 'relu',solver = solver)
    my_model.fc2.weight.data[0,:] = sol[:] 
    actual_neuron_list.append(my_model.fc1.bias.size(0)) 
    # compute the error 
    memory = 2**28 
    num_neuron = 0 if my_model == None else int(my_model.fc1.bias.detach().data.size(0))
    total_size2 = M*(num_neuron+1)
    num_batch2 = total_size2//memory + 1 
    batch_size_2 = M//num_batch2 # in
    
#     errl2 = (integration_weights.t()@(u_exact(integration_points) - my_model(integration_points).detach())**2)**0.5
    errl2 = compute_l2_error(u_exact,my_model,M,batch_size_2,integration_weights,integration_points)
#     errh1 = integration_weights.t()@(u_x(integration_points) - my_model.evaluate_derivative(integration_points,1).detach())**2
#     errh1 += integration_weights.t()@(u_y(integration_points) - my_model.evaluate_derivative(integration_points,2).detach())**2
#     errh1 += integration_weights.t()@(u_z(integration_points) - my_model.evaluate_derivative(integration_points,3).detach())**2 
#     errh1 = errh1**0.5    
    errh1 = compute_gradient_error(u_exact_grad,my_model,M,batch_size_2,integration_weights,integration_points)

    print("L2 error: ",errl2)
    print("H1 error: ",errh1) 
    err_list_l2.append(errl2.item())
    err_list_h1.append(errh1.item())
    # plot_2D(my_model.cpu())  

d = 3 
print("L2 order:",0.5 + (2*relu_k +1)/(2*d))
print("H1 order:",0.5 + (2*(relu_k-1) +1)/(2*d))

output_convergence_order(actual_neuron_list,err_list_l2, err_list_h1)

Number of neurons removed:  2
Number of neurons left:  23
total size: 23 421875 = 9703125
num batches:  1
assembling the mass matrix time taken:  0.38684988021850586
solving Ax = b time taken:  0.0004990100860595703
L2 error:  tensor(0.4197766976713)
H1 error:  tensor(1.5678911194972)
Number of neurons removed:  1
Number of neurons left:  49
total size: 49 421875 = 20671875
num batches:  1
assembling the mass matrix time taken:  1.5573410987854004
solving Ax = b time taken:  0.0004799365997314453
L2 error:  tensor(0.1281500448278)
H1 error:  tensor(0.7059528486438)
Number of neurons removed:  3
Number of neurons left:  97
total size: 97 421875 = 40921875
num batches:  1
assembling the mass matrix time taken:  2.1458170413970947
solving Ax = b time taken:  0.0005590915679931641
L2 error:  tensor(0.0325339470926)
H1 error:  tensor(0.2624441440502)
Number of neurons removed:  6
Number of neurons left:  194
total size: 194 421875 = 81843750
num batches:  1
assembling the mass matrix time t

- $u(x) = \Pi_{i=1}^3 (x_i^2 - 1)^2$

In [79]:
import sympy as sp
import torch
import numpy as np

# Define symbolic variables and the function u_exact
x_sym, y_sym, z_sym = sp.symbols('x y z')
u_expr = ((x_sym**2 - 1))**2 * ((y_sym**2 - 1))**2 * ((z_sym**2 - 1))**2

# Compute symbolic derivatives (e.g., first derivative with respect to x)
u_x_expr = sp.diff(u_expr, x_sym)
u_y_expr = sp.diff(u_expr, y_sym)
u_z_expr = sp.diff(u_expr, z_sym)
# Compute higher-order derivatives if needed:
u_xx_expr = sp.diff(u_expr, x_sym, 2)
u_yy_expr = sp.diff(u_expr, y_sym, 2)
u_zz_expr = sp.diff(u_expr, z_sym, 2)

# Convert the symbolic expressions to functions using lambdify (returns NumPy arrays)
u_exact_sym_func = sp.lambdify((x_sym, y_sym, z_sym), u_expr, modules='numpy')
u_x_sym_func     = sp.lambdify((x_sym, y_sym, z_sym), u_x_expr, modules='numpy')
u_y_sym_func     = sp.lambdify((x_sym, y_sym, z_sym), u_y_expr, modules='numpy')
u_z_sym_func     = sp.lambdify((x_sym, y_sym, z_sym), u_z_expr, modules='numpy')
u_xx_sym_func    = sp.lambdify((x_sym, y_sym, z_sym), u_xx_expr, modules='numpy')
u_yy_sym_func    = sp.lambdify((x_sym, y_sym, z_sym), u_yy_expr, modules='numpy')
u_zz_sym_func    = sp.lambdify((x_sym, y_sym, z_sym), u_zz_expr, modules='numpy')

# Define wrapper functions that accept PyTorch tensors as input and return torch tensors.
def u_exact(x_tensor):
    # Assume x_tensor is a tensor of shape (N, 2) where each row is (x, y)
    x_np = x_tensor.detach().cpu().numpy()
    # Evaluate the symbolic function using the first and second columns
    result_np = u_exact_sym_func(x_np[:, 0], x_np[:, 1], x_np[:, 2]).reshape(-1,1)
    # Convert result to a torch tensor, preserving the device and dtype of the input
    return torch.from_numpy(np.array(result_np)).to(x_tensor.device).type(x_tensor.dtype)

def u_x(x_tensor):
    x_np = x_tensor.detach().cpu().numpy()
    result_np = u_x_sym_func(x_np[:, 0], x_np[:, 1], x_np[:, 2]).reshape(-1,1)
    return torch.from_numpy(np.array(result_np)).to(x_tensor.device).type(x_tensor.dtype)

def u_y(x_tensor):
    x_np = x_tensor.detach().cpu().numpy()
    result_np = u_y_sym_func(x_np[:, 0], x_np[:, 1], x_np[:, 2] ).reshape(-1,1)
    return torch.from_numpy(np.array(result_np)).to(x_tensor.device).type(x_tensor.dtype)

def u_z(x_tensor):
    x_np = x_tensor.detach().cpu().numpy()
    result_np = u_z_sym_func(x_np[:, 0], x_np[:, 1], x_np[:, 2] ).reshape(-1,1)
    return torch.from_numpy(np.array(result_np)).to(x_tensor.device).type(x_tensor.dtype)

def u_xx(x_tensor):
    x_np = x_tensor.detach().cpu().numpy()
    result_np = u_xx_sym_func(x_np[:, 0], x_np[:, 1], x_np[:, 2]).reshape(-1,1)
    return torch.from_numpy(np.array(result_np)).to(x_tensor.device).type(x_tensor.dtype)

def u_yy(x_tensor):
    x_np = x_tensor.detach().cpu().numpy()
    result_np = u_yy_sym_func(x_np[:, 0], x_np[:, 1], x_np[:, 2]).reshape(-1,1)
    return torch.from_numpy(np.array(result_np)).to(x_tensor.device).type(x_tensor.dtype)

def u_zz(x_tensor):
    x_np = x_tensor.detach().cpu().numpy()
    result_np = u_zz_sym_func(x_np[:, 0], x_np[:, 1], x_np[:, 2]).reshape(-1,1)
    return torch.from_numpy(np.array(result_np)).to(x_tensor.device).type(x_tensor.dtype)

def target(x_tensor):
    return - u_xx(x_tensor) - u_yy(x_tensor) -u_zz(x_tensor) + u_exact(x_tensor) 
def u_exact_grad(): 
    return [u_x, u_y, u_z]

Nx = 50    
order = 3 
integration_weights, integration_points = PiecewiseGQ3D_weights_points(Nx, order,[-1,-1,-1],[1,1,1])
M = integration_points.size(0)
neuron_num_list = [25,50,100,200,400,800,1600]
actual_neuron_list = []
err_list_l2 = []
err_list_h1 = []
relu_k = 3 
for neuron_num in neuron_num_list: 
    my_model = model(3,neuron_num,1,relu_k).to(device)
    my_model = initialize_model_2(my_model,dim=3).to(device)
    my_model = remove_redundant_neuron(my_model.cpu(), dims = 3, choice = 2).to(device) 
    solver = 'direct'
    # sol = minimize_linear_layer_H1_explicit_assemble_efficient(my_model,target,integration_weights, integration_points,activation = 'relu',solver = solver)
    sol = minimize_linear_layer_H1_explicit_assemble_efficient_general_dim(my_model,target,None,integration_weights, integration_points,torch.tensor([]),torch.tensor([]),activation = 'relu',solver = solver,memory=2**27)
    my_model.fc2.weight.data[0,:] = sol[:] 
    actual_neuron_list.append(my_model.fc1.bias.size(0)) 
    
    # compute the error 
    memory = 2**27 
    num_neuron = 0 if my_model == None else int(my_model.fc1.bias.detach().data.size(0))
    total_size2 = M*(num_neuron+1)
    num_batch2 = total_size2//memory + 1 
    batch_size_2 = M//num_batch2 # in
    
#     errl2 = (integration_weights.t()@(u_exact(integration_points) - my_model(integration_points).detach())**2)**0.5
    errl2 = compute_l2_error(u_exact,my_model,M,batch_size_2,integration_weights,integration_points)
#     errh1 = integration_weights.t()@(u_x(integration_points) - my_model.evaluate_derivative(integration_points,1).detach())**2
#     errh1 += integration_weights.t()@(u_y(integration_points) - my_model.evaluate_derivative(integration_points,2).detach())**2
#     errh1 += integration_weights.t()@(u_z(integration_points) - my_model.evaluate_derivative(integration_points,3).detach())**2 
#     errh1 = errh1**0.5    
    errh1 = compute_gradient_error(u_exact_grad,my_model,M,batch_size_2,integration_weights,integration_points)
    print("L2 error: ",errl2)
    print("H1 error: ",errh1) 
    err_list_l2.append(errl2.item())
    err_list_h1.append(errh1.item())
    # plot_2D(my_model.cpu())  

Number of neurons removed:  2
Number of neurons left:  23
total size: 23 3375000 = 77625000
num batches:  1
assembling the mass matrix time taken:  0.13536524772644043
solving Ax = b time taken:  0.01958918571472168
L2 error:  tensor(0.9820892295028, device='cuda:0')
H1 error:  tensor(1.8463127648653, device='cuda:0')
Number of neurons removed:  1
Number of neurons left:  49
total size: 49 3375000 = 165375000
num batches:  2
assembling the mass matrix time taken:  0.10634565353393555
solving Ax = b time taken:  0.037626028060913086
L2 error:  tensor(0.1780856821902, device='cuda:0')
H1 error:  tensor(1.0071909102789, device='cuda:0')
Number of neurons removed:  3
Number of neurons left:  97
total size: 97 3375000 = 327375000
num batches:  3
assembling the mass matrix time taken:  0.11069560050964355
solving Ax = b time taken:  0.07927560806274414
L2 error:  tensor(0.0599353074740, device='cuda:0')
H1 error:  tensor(0.4590523845964, device='cuda:0')
Number of neurons removed:  9
Number 

In [80]:
d = 3 
print("L2 order:",0.5 + (2*relu_k +1)/(2*d))
print("H1 order:",0.5 + (2*(relu_k-1) +1)/(2*d))

output_convergence_order(actual_neuron_list,err_list_l2, err_list_h1)

L2 order: 1.6666666666666667
H1 order: 1.3333333333333335
neuron num  & 	 $\|u-u_n \|_{L^2}$ & 	 order  & $ |u-u_n |_{H^1}$ & 	 order\\ \hline \hline 
23 		 & 9.821e-01 &		 * & 1.846e+00 &  *  \\ \hline  

49 		 &  1.781e-01 &  		 2.26  &  1.007e+00 &  0.80  \\ \hline  

97 		 &  5.994e-02 &  		 1.59  &  4.591e-01 &  1.15  \\ \hline  

191 		 &  2.194e-02 &  		 1.48  &  2.325e-01 &  1.00  \\ \hline  

382 		 &  7.129e-03 &  		 1.62  &  9.841e-02 &  1.24  \\ \hline  

750 		 &  2.799e-03 &  		 1.39  &  4.759e-02 &  1.08  \\ \hline  

1483 		 &  9.589e-04 &  		 1.57  &  2.053e-02 &  1.23  \\ \hline  



In [78]:
d = 3 
print("L2 order:",0.5 + (2*relu_k +1)/(2*d))
print("H1 order:",0.5 + (2*(relu_k-1) +1)/(2*d))

output_convergence_order(actual_neuron_list,err_list_l2, err_list_h1)

L2 order: 1.6666666666666667
H1 order: 1.3333333333333335
neuron num  & 	 $\|u-u_n \|_{L^2}$ & 	 order  & $ |u-u_n |_{H^1}$ & 	 order\\ \hline \hline 
24 		 & 4.598e-01 &		 * & 1.866e+00 &  *  \\ \hline  

48 		 &  1.727e-01 &  		 1.41  &  9.884e-01 &  0.92  \\ \hline  

99 		 &  6.589e-02 &  		 1.33  &  5.049e-01 &  0.93  \\ \hline  

193 		 &  1.863e-02 &  		 1.89  &  1.966e-01 &  1.41  \\ \hline  

384 		 &  6.899e-03 &  		 1.44  &  9.480e-02 &  1.06  \\ \hline  

744 		 &  2.910e-03 &  		 1.31  &  4.871e-02 &  1.01  \\ \hline  



## 4D example



In [17]:
##1. exact solution: sin(pi/2 x)sin(pi/2 y) 
def u_exact(x): 
    z = torch.sin(pi/2*x[:,0:1]) * torch.sin(pi/2*x[:,1:2] ) * torch.sin(pi/2*x[:,2:3]) * torch.sin(pi/2*x[:,3:4])   
    return z 
def u_x(x):
    z = (pi/2)*torch.cos(pi/2*x[:,0:1])*torch.sin(pi/2*x[:,1:2] ) * torch.sin(pi/2*x[:,2:3]) * torch.sin(pi/2*x[:,3:4])  
    return z
def u_y(x):
    z = (pi/2)*torch.sin(pi/2*x[:,0:1])*torch.cos(pi/2*x[:,1:2] ) * torch.sin(pi/2*x[:,2:3]) * torch.sin(pi/2*x[:,3:4])  
    return z 
def u_z(x):
    z = (pi/2)*torch.sin(pi/2*x[:,0:1])*torch.sin(pi/2*x[:,1:2] ) * torch.cos(pi/2*x[:,2:3]) * torch.sin(pi/2*x[:,3:4])  
    return z 
def u_4(x):
    z = (pi/2)*torch.sin(pi/2*x[:,0:1])*torch.sin(pi/2*x[:,1:2] ) * torch.sin(pi/2*x[:,2:3]) * torch.cos(pi/2*x[:,3:4])  
    return z

def target(x):
    z = (4 * (pi/2)**2 + 1)*torch.sin(pi/2*x[:,0:1])*torch.sin(pi/2*x[:,1:2] ) * torch.sin(pi/2*x[:,2:3]) * torch.sin(pi/2*x[:,3:4])  
    return z

def u_exact_grad():
    return [u_x, u_y,u_z,u_4]


M = int(1e6) 
integration_weights, integration_points = MonteCarlo_Sobol_dDim_weights_points(M, 4, -1, 1) 

neuron_num_list = [25,50,100,200,400,800,1600]
actual_neuron_list = []
err_list_l2 = []
err_list_h1 = []
relu_k = 3 
for neuron_num in neuron_num_list: 
    my_model = model(4,neuron_num,1,relu_k).to(device)
    my_model = initialize_model_2(my_model,dim=4).to(device)
    my_model = remove_redundant_neuron(my_model.cpu(), dims = 4, choice = 2).to(device) 
    solver = 'direct'
    # sol = minimize_linear_layer_H1_explicit_assemble_efficient(my_model,target,integration_weights, integration_points,activation = 'relu',solver = solver)
    sol = minimize_linear_layer_H1_explicit_assemble_efficient_general_dim(my_model,target,None,integration_weights, integration_points,torch.tensor([]),torch.tensor([]),activation = 'relu',solver = solver,memory = 2**27)
    my_model.fc2.weight.data[0,:] = sol[:] 
    actual_neuron_list.append(my_model.fc1.bias.size(0)) 
    # compute the error 
    # compute the error 
    memory = 2**28 
    num_neuron = 0 if my_model == None else int(my_model.fc1.bias.detach().data.size(0))
    total_size2 = M*(num_neuron+1)
    num_batch2 = total_size2//memory + 1 
    batch_size_2 = M//num_batch2 #
    
#     errl2 = (integration_weights.t()@(u_exact(integration_points) - my_model(integration_points).detach())**2)**0.5
    errl2 = compute_l2_error(u_exact,my_model,M,batch_size_2,integration_weights,integration_points)
#     errh1 = integration_weights.t()@(u_x(integration_points) - my_model.evaluate_derivative(integration_points,1).detach())**2
#     errh1 += integration_weights.t()@(u_y(integration_points) - my_model.evaluate_derivative(integration_points,2).detach())**2
#     errh1 += integration_weights.t()@(u_z(integration_points) - my_model.evaluate_derivative(integration_points,3).detach())**2 
#     errh1 = errh1**0.5    
    errh1 = compute_gradient_error(u_exact_grad,my_model,M,batch_size_2,integration_weights,integration_points)

    print("L2 error: ",errl2)
    print("H1 error: ",errh1) 
    err_list_l2.append(errl2.item())
    err_list_h1.append(errh1.item())
    # plot_2D(my_model.cpu())  

Number of neurons removed:  1
Number of neurons left:  24
total size: 24 1000000 = 24000000
num batches:  1
assembling the mass matrix time taken:  0.0011906623840332031
solving Ax = b time taken:  0.009128332138061523
L2 error:  tensor(0.9557047407699, device='cuda:0')
H1 error:  tensor(3.0245912129283, device='cuda:0')
Number of neurons removed:  1
Number of neurons left:  49
total size: 49 1000000 = 49000000
num batches:  1
assembling the mass matrix time taken:  0.0009484291076660156
solving Ax = b time taken:  0.01781487464904785
L2 error:  tensor(0.7325393237861, device='cuda:0')
H1 error:  tensor(2.4500431558857, device='cuda:0')
Number of neurons removed:  5
Number of neurons left:  95
total size: 95 1000000 = 95000000
num batches:  1
assembling the mass matrix time taken:  0.0009667873382568359
solving Ax = b time taken:  0.03963804244995117
L2 error:  tensor(0.3122166515614, device='cuda:0')
H1 error:  tensor(1.5002636550614, device='cuda:0')
Number of neurons removed:  1
Num

In [18]:
d = 4  
print("L2 order:",0.5 + (2*relu_k +1)/(2*d))
print("H1 order:",0.5 + (2*(relu_k-1) +1)/(2*d))

output_convergence_order(actual_neuron_list,err_list_l2, err_list_h1)

L2 order: 1.375
H1 order: 1.125
$n$ & 	 $\|u-u_n \|_{L^2}$ & 	 order  & $ |u-u_n |_{H^1}$ & 	 order\\ \hline \hline 
24 		 & 9.557e-01 &		 * & 3.025e+00 &  *  \\ \hline  

49 		 &  7.325e-01 &  		 0.37  &  2.450e+00 &  0.30  \\ \hline  

95 		 &  3.122e-01 &  		 1.29  &  1.500e+00 &  0.74  \\ \hline  

199 		 &  9.747e-02 &  		 1.57  &  6.570e-01 &  1.12  \\ \hline  

393 		 &  3.315e-02 &  		 1.58  &  2.883e-01 &  1.21  \\ \hline  

791 		 &  1.109e-02 &  		 1.57  &  1.190e-01 &  1.27  \\ \hline  

1575 		 &  4.142e-03 &  		 1.43  &  5.322e-02 &  1.17  \\ \hline  



 - $u(x) = \Pi_{i=1}^4 (x_i^2 - 1)^2$

In [20]:
import sympy as sp
import torch
import numpy as np

# Define symbolic variables and the function u_exact
x1_sym, x2_sym, x3_sym, x4_sym = sp.symbols('x1 x2 x3 x4')
u_expr = ((x1_sym**2 - 1))**2 * ((x2_sym**2 - 1))**2 * ((x3_sym**2 - 1))**2 * ((x4_sym**2 - 1))**2

# Compute symbolic derivatives (e.g., first derivative with respect to x)
u_x1_expr = sp.diff(u_expr, x1_sym)
u_x2_expr = sp.diff(u_expr, x2_sym)
u_x3_expr = sp.diff(u_expr, x3_sym)
u_x4_expr = sp.diff(u_expr, x4_sym)
# Compute higher-order derivatives if needed:
u_11_expr = sp.diff(u_expr, x1_sym, 2)
u_22_expr = sp.diff(u_expr, x2_sym, 2)
u_33_expr = sp.diff(u_expr, x3_sym, 2)
u_44_expr = sp.diff(u_expr, x4_sym, 2)

# Convert the symbolic expressions to functions using lambdify (returns NumPy arrays)
u_exact_sym_func = sp.lambdify((x1_sym, x2_sym, x3_sym, x4_sym), u_expr, modules='numpy')
u_x1_sym_func     = sp.lambdify((x1_sym, x2_sym, x3_sym, x4_sym), u_x1_expr, modules='numpy')
u_x2_sym_func     = sp.lambdify((x1_sym, x2_sym, x3_sym, x4_sym), u_x2_expr, modules='numpy')
u_x3_sym_func     = sp.lambdify((x1_sym, x2_sym, x3_sym, x4_sym), u_x3_expr, modules='numpy')
u_x4_sym_func     = sp.lambdify((x1_sym, x2_sym, x3_sym, x4_sym), u_x4_expr, modules='numpy')

u_11_sym_func    = sp.lambdify((x1_sym, x2_sym, x3_sym, x4_sym), u_11_expr, modules='numpy')
u_22_sym_func    = sp.lambdify((x1_sym, x2_sym, x3_sym, x4_sym), u_22_expr, modules='numpy')
u_33_sym_func    = sp.lambdify((x1_sym, x2_sym, x3_sym, x4_sym), u_33_expr, modules='numpy')
u_44_sym_func    = sp.lambdify((x1_sym, x2_sym, x3_sym, x4_sym), u_44_expr, modules='numpy')


# Define wrapper functions that accept PyTorch tensors as input and return torch tensors.
def u_exact(x_tensor):
    # Assume x_tensor is a tensor of shape (N, 2) where each row is (x, y)
    x_np = x_tensor.detach().cpu().numpy()
    # Evaluate the symbolic function using the first and second columns
    result_np = u_exact_sym_func(x_np[:, 0], x_np[:, 1], x_np[:, 2],x_np[:, 3]).reshape(-1,1)
    # Convert result to a torch tensor, preserving the device and dtype of the input
    return torch.from_numpy(np.array(result_np)).to(x_tensor.device).type(x_tensor.dtype)

def u_x1(x_tensor):
    x_np = x_tensor.detach().cpu().numpy()
    result_np = u_x1_sym_func(x_np[:, 0], x_np[:, 1], x_np[:, 2],x_np[:, 3]).reshape(-1,1)
    return torch.from_numpy(np.array(result_np)).to(x_tensor.device).type(x_tensor.dtype)

def u_x2(x_tensor):
    x_np = x_tensor.detach().cpu().numpy()
    result_np = u_x2_sym_func(x_np[:, 0], x_np[:, 1], x_np[:, 2],x_np[:, 3]).reshape(-1,1)
    return torch.from_numpy(np.array(result_np)).to(x_tensor.device).type(x_tensor.dtype)

def u_x3(x_tensor):
    x_np = x_tensor.detach().cpu().numpy()
    result_np = u_x3_sym_func(x_np[:, 0], x_np[:, 1], x_np[:, 2],x_np[:, 3]).reshape(-1,1)
    return torch.from_numpy(np.array(result_np)).to(x_tensor.device).type(x_tensor.dtype)

def u_x4(x_tensor):
    x_np = x_tensor.detach().cpu().numpy()
    result_np = u_x4_sym_func(x_np[:, 0], x_np[:, 1], x_np[:, 2],x_np[:, 3]).reshape(-1,1)
    return torch.from_numpy(np.array(result_np)).to(x_tensor.device).type(x_tensor.dtype)

def u_11(x_tensor):
    x_np = x_tensor.detach().cpu().numpy()
    result_np = u_11_sym_func(x_np[:, 0], x_np[:, 1], x_np[:, 2],x_np[:, 3]).reshape(-1,1)
    return torch.from_numpy(np.array(result_np)).to(x_tensor.device).type(x_tensor.dtype)

def u_22(x_tensor):
    x_np = x_tensor.detach().cpu().numpy()
    result_np = u_22_sym_func(x_np[:, 0], x_np[:, 1], x_np[:, 2],x_np[:, 3]).reshape(-1,1)
    return torch.from_numpy(np.array(result_np)).to(x_tensor.device).type(x_tensor.dtype)

def u_33(x_tensor):
    x_np = x_tensor.detach().cpu().numpy()
    result_np = u_33_sym_func(x_np[:, 0], x_np[:, 1], x_np[:, 2],x_np[:, 3]).reshape(-1,1)
    return torch.from_numpy(np.array(result_np)).to(x_tensor.device).type(x_tensor.dtype)

def u_44(x_tensor):
    x_np = x_tensor.detach().cpu().numpy()
    result_np = u_44_sym_func(x_np[:, 0], x_np[:, 1], x_np[:, 2],x_np[:, 3]).reshape(-1,1)
    return torch.from_numpy(np.array(result_np)).to(x_tensor.device).type(x_tensor.dtype)

def target(x_tensor):
    return - u_11(x_tensor) - u_22(x_tensor) -u_33(x_tensor) - u_44(x_tensor) + u_exact(x_tensor) 
def u_exact_grad(): 
    return [u_x1, u_x2, u_x3, u_x4]


M = int(1e7) 
integration_weights, integration_points = MonteCarlo_Sobol_dDim_weights_points(M, 4, -1, 1) 

neuron_num_list = [25,50,100,200,400,800,1600]
actual_neuron_list = []
err_list_l2 = []
err_list_h1 = []
relu_k = 4 
for neuron_num in neuron_num_list: 
    my_model = model(4,neuron_num,1,relu_k).to(device)
    my_model = initialize_model_2(my_model,dim=4).to(device)
    my_model = remove_redundant_neuron(my_model.cpu(), dims = 4, choice = 2).to(device) 
    solver = 'direct'
    # sol = minimize_linear_layer_H1_explicit_assemble_efficient(my_model,target,integration_weights, integration_points,activation = 'relu',solver = solver)
    sol = minimize_linear_layer_H1_explicit_assemble_efficient_general_dim(my_model,target,None,integration_weights, integration_points,torch.tensor([]),torch.tensor([]),activation = 'relu',solver = solver,memory = 2**27)
    my_model.fc2.weight.data[0,:] = sol[:] 
    actual_neuron_list.append(my_model.fc1.bias.size(0)) 
    # compute the error 
    # compute the error 
    memory = 2**28
    num_neuron = 0 if my_model == None else int(my_model.fc1.bias.detach().data.size(0))
    total_size2 = M*(num_neuron+1)
    num_batch2 = total_size2//memory + 1 
    batch_size_2 = M//num_batch2 #
    
#     errl2 = (integration_weights.t()@(u_exact(integration_points) - my_model(integration_points).detach())**2)**0.5
    errl2 = compute_l2_error(u_exact,my_model,M,batch_size_2,integration_weights,integration_points)
#     errh1 = integration_weights.t()@(u_x(integration_points) - my_model.evaluate_derivative(integration_points,1).detach())**2
#     errh1 += integration_weights.t()@(u_y(integration_points) - my_model.evaluate_derivative(integration_points,2).detach())**2
#     errh1 += integration_weights.t()@(u_z(integration_points) - my_model.evaluate_derivative(integration_points,3).detach())**2 
#     errh1 = errh1**0.5    
    errh1 = compute_gradient_error(u_exact_grad,my_model,M,batch_size_2,integration_weights,integration_points)

    print("L2 error: ",errl2)
    print("H1 error: ",errh1) 
    err_list_l2.append(errl2.item())
    err_list_h1.append(errh1.item())
    # plot_2D(my_model.cpu())  


Number of neurons removed:  1
Number of neurons left:  24
total size: 24 10000000 = 240000000
num batches:  2
assembling the mass matrix time taken:  0.7632627487182617
solving Ax = b time taken:  0.07711434364318848
L2 error:  tensor(0.7429277569460, device='cuda:0')
H1 error:  tensor(2.1546242931617, device='cuda:0')
Number of neurons removed:  3
Number of neurons left:  47
total size: 47 10000000 = 470000000
num batches:  4
assembling the mass matrix time taken:  0.6694808006286621
solving Ax = b time taken:  0.1373908519744873
L2 error:  tensor(0.8510203186632, device='cuda:0')
H1 error:  tensor(2.0509476719127, device='cuda:0')
Number of neurons removed:  2
Number of neurons left:  98
total size: 98 10000000 = 980000000
num batches:  8
assembling the mass matrix time taken:  0.5800361633300781
solving Ax = b time taken:  0.3305537700653076
L2 error:  tensor(0.5831090559255, device='cuda:0')
H1 error:  tensor(1.8432036461139, device='cuda:0')
Number of neurons removed:  3
Number of

In [21]:
d = 4  
print("L2 order:",0.5 + (2*relu_k +1)/(2*d))
print("H1 order:",0.5 + (2*(relu_k-1) +1)/(2*d))

output_convergence_order(actual_neuron_list,err_list_l2, err_list_h1)

L2 order: 1.625
H1 order: 1.375
neuron num  & 	 $\|u-u_n \|_{L^2}$ & 	 order  & $ |u-u_n |_{H^1}$ & 	 order\\ \hline \hline 
24 		 & 7.429e-01 &		 * & 2.155e+00 &  *  \\ \hline  

47 		 &  8.510e-01 &  		 -0.20  &  2.051e+00 &  0.07  \\ \hline  

98 		 &  5.831e-01 &  		 0.51  &  1.843e+00 &  0.15  \\ \hline  

197 		 &  1.135e-01 &  		 2.34  &  7.660e-01 &  1.26  \\ \hline  

394 		 &  4.469e-02 &  		 1.34  &  3.930e-01 &  0.96  \\ \hline  

790 		 &  1.433e-02 &  		 1.64  &  1.636e-01 &  1.26  \\ \hline  

1583 		 &  6.210e-03 &  		 1.20  &  8.406e-02 &  0.96  \\ \hline  



## 5D example

In [23]:
def u_exact(x):
    z = torch.prod(torch.sin(pi/2 * x),dim = 1,keepdim = True)
    return z 

def u_1(x):
    sin_terms = torch.sin(pi/2 * x)
    cos_term = torch.cos(pi/2 * x[:, 0:1])
    sin_terms[:, 0:1] = cos_term  # Replace the first sine term with the cosine term
    z = (pi/2) * torch.prod(sin_terms, dim=1, keepdim=True)
    return z 

def u_2(x):
    sin_terms = torch.sin(pi/2 * x)
    cos_term = torch.cos(pi/2 * x[:, 1:2])
    sin_terms[:, 1:2] = cos_term  # Replace the second sine term with the cosine term
    z = (pi/2) * torch.prod(sin_terms, dim=1, keepdim=True)
    return z

def u_3(x):
    sin_terms = torch.sin(pi/2 * x)
    cos_term = torch.cos(pi/2 * x[:, 2:3])
    sin_terms[:, 2:3] = cos_term  # Replace the third sine term with the cosine term
    z = (pi/2) * torch.prod(sin_terms, dim=1, keepdim=True)
    return z

def u_4(x):
    sin_terms = torch.sin(pi/2 * x)
    cos_term = torch.cos(pi/2 * x[:, 3:4])
    sin_terms[:, 3:4] = cos_term  # Replace the fourth sine term with the cosine term
    z = (pi/2) * torch.prod(sin_terms, dim=1, keepdim=True)
    return z

def u_5(x):
    sin_terms = torch.sin(pi/2 * x)
    cos_term = torch.cos(pi/2 * x[:, 4:5])
    sin_terms[:, 4:5] = cos_term  # Replace the fifth sine term with the cosine term
    z = (pi/2) * torch.prod(sin_terms, dim=1, keepdim=True)
    return z

def target(x):
    z = (5 * (pi/2)**2 + 1)*torch.prod(torch.sin(pi/2 * x),dim = 1,keepdim = True) 
    return z

def u_exact_grad():
    return [u_1, u_2,u_3,u_4,u_5]


M = int(6e6) 
integration_weights, integration_points = MonteCarlo_Sobol_dDim_weights_points(M, 5, -1, 1) 

neuron_num_list = [25,50,100,200,400,800,1600,3200]
actual_neuron_list = []
err_list_l2 = []
err_list_h1 = []
relu_k = 3  
for neuron_num in neuron_num_list: 
    my_model = model(5,neuron_num,1,relu_k).to(device)
    my_model = initialize_model_2(my_model,dim=5).to(device)
    my_model = remove_redundant_neuron(my_model.cpu(), dims = 5, choice = 2).to(device) 
    solver = 'direct'
    # sol = minimize_linear_layer_H1_explicit_assemble_efficient(my_model,target,integration_weights, integration_points,activation = 'relu',solver = solver)
    sol = minimize_linear_layer_H1_explicit_assemble_efficient_general_dim(my_model,target,None,integration_weights, integration_points,torch.tensor([]),torch.tensor([]),activation = 'relu',solver = solver,memory = 2**27)
    my_model.fc2.weight.data[0,:] = sol[:] 
    actual_neuron_list.append(my_model.fc1.bias.size(0)) 
    # compute the error 
    # compute the error 
    memory = 2**28
    num_neuron = 0 if my_model == None else int(my_model.fc1.bias.detach().data.size(0))
    total_size2 = M*(num_neuron+1)
    num_batch2 = total_size2//memory + 1 
    batch_size_2 = M//num_batch2 #
    
#     errl2 = (integration_weights.t()@(u_exact(integration_points) - my_model(integration_points).detach())**2)**0.5
    errl2 = compute_l2_error(u_exact,my_model,M,batch_size_2,integration_weights,integration_points)
#     errh1 = integration_weights.t()@(u_x(integration_points) - my_model.evaluate_derivative(integration_points,1).detach())**2
#     errh1 += integration_weights.t()@(u_y(integration_points) - my_model.evaluate_derivative(integration_points,2).detach())**2
#     errh1 += integration_weights.t()@(u_z(integration_points) - my_model.evaluate_derivative(integration_points,3).detach())**2 
#     errh1 = errh1**0.5    
    errh1 = compute_gradient_error(u_exact_grad,my_model,M,batch_size_2,integration_weights,integration_points)

    print("L2 error: ",errl2)
    print("H1 error: ",errh1) 
    err_list_l2.append(errl2.item())
    err_list_h1.append(errh1.item())
    # plot_2D(my_model.cpu())  

Number of neurons removed:  0
Number of neurons left:  25
total size: 25 6000000 = 150000000
num batches:  2
assembling the mass matrix time taken:  0.002214670181274414
solving Ax = b time taken:  0.07104921340942383
L2 error:  tensor(0.9967371607577, device='cuda:0')
H1 error:  tensor(3.5015063629117, device='cuda:0')
Number of neurons removed:  0
Number of neurons left:  50
total size: 50 6000000 = 300000000
num batches:  3
assembling the mass matrix time taken:  0.0027451515197753906
solving Ax = b time taken:  0.12767958641052246
L2 error:  tensor(0.9793662018857, device='cuda:0')
H1 error:  tensor(3.4554203706355, device='cuda:0')
Number of neurons removed:  1
Number of neurons left:  99
total size: 99 6000000 = 594000000
num batches:  5
assembling the mass matrix time taken:  0.004477262496948242
solving Ax = b time taken:  0.2763504981994629
L2 error:  tensor(0.9353335207245, device='cuda:0')
H1 error:  tensor(3.3563104219453, device='cuda:0')
Number of neurons removed:  1
Numb

In [24]:
d = 5   
print("L2 order:",0.5 + (2*relu_k +1)/(2*d))
print("H1 order:",0.5 + (2*(relu_k-1) +1)/(2*d))

output_convergence_order(actual_neuron_list,err_list_l2, err_list_h1)

L2 order: 1.2
H1 order: 1.0
$n$ & 	 $\|u-u_n \|_{L^2}$ & 	 order  & $ |u-u_n |_{H^1}$ & 	 order\\ \hline \hline 
25 		 & 9.967e-01 &		 * & 3.502e+00 &  *  \\ \hline  

50 		 &  9.794e-01 &  		 0.03  &  3.455e+00 &  0.02  \\ \hline  

99 		 &  9.353e-01 &  		 0.07  &  3.356e+00 &  0.04  \\ \hline  

199 		 &  6.036e-01 &  		 0.63  &  2.533e+00 &  0.40  \\ \hline  

399 		 &  2.408e-01 &  		 1.32  &  1.367e+00 &  0.89  \\ \hline  

798 		 &  7.664e-02 &  		 1.65  &  5.802e-01 &  1.24  \\ \hline  

1590 		 &  3.730e-02 &  		 1.04  &  3.400e-01 &  0.78  \\ \hline  

3182 		 &  1.582e-02 &  		 1.24  &  1.751e-01 &  0.96  \\ \hline  



## 6D example 

In [ ]:

def compute_l2_error(u_exact,my_model,M,batch_size_2,weights,integration_points): 
    err = 0 
    if my_model == None: 
        for jj in range(0,M,batch_size_2): 
            end_index = jj + batch_size_2 
            func_values = u_exact(integration_points[jj:end_index,:])
            err += torch.sum(func_values**2 * weights[jj:end_index,:])
    else: 
        for jj in range(0,M,batch_size_2): 
            end_index = jj + batch_size_2 
            func_values = u_exact(integration_points[jj:end_index,:]) - my_model(integration_points[jj:end_index,:]).detach()
            err += torch.sum(func_values**2 * weights[jj:end_index,:])
    return err**0.5 

def compute_gradient_error(u_exact_grad,my_model,M,batch_size_2,weights,integration_points):
    """
    Parameters
    ----------
    u_exact_grad: list or None
        a list that contains ways of evaluating partial derivatives that gives the gradient  
    """
    err_h10 = 0 
     # initial gradient error 
    if u_exact_grad != None and my_model!=None:
        u_grad = u_exact_grad() 
        for ii, grad_i in enumerate(u_grad): 
            for jj in range(0,M,batch_size_2): 
                end_index = jj + batch_size_2 
                my_model_dxi = my_model.evaluate_derivative(integration_points[jj:end_index,:],ii+1).detach() 
                err_h10 += torch.sum((grad_i(integration_points[jj:end_index,:]) - my_model_dxi)**2 * weights[jj:end_index,:])
    elif u_exact_grad != None and my_model==None:
        u_grad = u_exact_grad() 
        for grad_i in u_grad: 
            for jj in range(0,M,batch_size_2): 
                end_index = jj + batch_size_2 
                err_h10 += torch.sum((grad_i(integration_points[jj:end_index,:]))**2 * weights[jj:end_index,:])
    return err_h10**0.5

In [ ]:


def minimize_linear_layer_H1_explicit_assemble_efficient_general_dim(model,target, g_N, weights, integration_points, w_bd, pts_bd, activation = 'relu',solver="direct",memory = 2**29 ):
    """ -div alpha grad u(x) + u = f 
    Parameters
    ----------
    model: 
        nn model
    alpha:
        alpha function
    target:
        rhs function f 
    pts_bd:
        integration points on the boundary, embdedded in the domain 
    """ 
    zero = torch.tensor([0.]).to(device)
    start_time = time.time() 
    w = model.fc1.weight.data 
    b = model.fc1.bias.data 
    neuron_num = b.size(0) 
    dim = integration_points.size(1) 
    M = integration_points.size(0)

    total_size = neuron_num * M # memory, number of floating numbers 
    print('total size: {} {} = {}'.format(neuron_num,M,total_size))
    num_batch = total_size//memory + 1 # divide according to memory
    print("num batches: ",num_batch)
    batch_size = M//num_batch
    start_ind = 0
    end_ind = 0 
    jac = torch.zeros(b.size(0),b.size(0)).to(device)
    rhs = torch.zeros(b.size(0),1).to(device)

    for j in range(0,M,batch_size): # batch operation in data points 
        end_ind = j + batch_size
        basis_value_col = F.relu(integration_points[j:end_ind] @ w.t()+ b)**(model.k) 
        weighted_basis_value_col = basis_value_col * weights[j:end_ind] 
        jac += weighted_basis_value_col.t() @ basis_value_col 
        rhs += weighted_basis_value_col.t() @ (target(integration_points[j:end_ind,:])) 

    # Assemble the boundary condition term <g,v>_{\Gamma_N} 
    size_pts_bd = int(pts_bd.size(0)/(2*dim))
    # M_bc = size_pts_bd 
    # total_size = M_bc * neuron_num 
    # num_batch = total_size//memory + 1 
    # batch_size = M_bc//num_batch
    if g_N != None:
        bcs_N = g_N(dim)
        for ii, g_ii in bcs_N:
            weighted_g_N = -g_ii(pts_bd[2*ii*size_pts_bd:(2*ii+1)*size_pts_bd,:])* w_bd[2*ii*size_pts_bd:(2*ii+1)*size_pts_bd,:]
            basis_value_bd_col = F.relu(pts_bd[2*ii*size_pts_bd:(2*ii+1)*size_pts_bd,:] @ w.t()+ b)**(model.k)
            rhs += basis_value_bd_col.t() @ weighted_g_N

            weighted_g_N = g_ii(pts_bd[(2*ii+1)*size_pts_bd:(2*ii+2)*size_pts_bd,:])* w_bd[(2*ii+1)*size_pts_bd:(2*ii+2)*size_pts_bd,:]
            basis_value_bd_col = F.relu(pts_bd[(2*ii+1)*size_pts_bd:(2*ii+2)*size_pts_bd,:] @ w.t()+ b)**(model.k)
            rhs += basis_value_bd_col.t() @ weighted_g_N
            
    # Stiffness matrix term in the jacobian 
    for d in range(dim):
        end_ind = 0 
        if model.k == 1:  
            for j in range(0,M,batch_size): 
                end_ind = j + batch_size 
                basis_value_dxi_col = torch.heaviside(integration_points[j:end_ind] @ w.t()+ b, zero) * w.t()[d:d+1,:]
                weighted_basis_value_dx_col = basis_value_dxi_col * weights[j:end_ind] 
                jac += weighted_basis_value_dx_col.t() @ basis_value_dxi_col 
#             basis_value_dxi_col = torch.heaviside(integration_points @ w.t()+ b, zero) * w.t()[d:d+1,:]
#             weighted_basis_value_dx_col = basis_value_dxi_col * weights * coef_alpha 
#             jac += weighted_basis_value_dx_col.t() @ basis_value_dxi_col 

        else: 
            for j in range(0,M,batch_size):  
                end_ind = j + batch_size 
                basis_value_dxi_col = model.k * F.relu(integration_points[j:end_ind] @ w.t()+ b)**(model.k-1) * w.t()[d:d+1,:]
                weighted_basis_value_dx_col = basis_value_dxi_col * weights[j:end_ind]  
                jac += weighted_basis_value_dx_col.t() @ basis_value_dxi_col 
#             basis_value_dxi_col = model.k * F.relu(integration_points @ w.t()+ b)**(model.k-1) * w.t()[d:d+1,:]
#             weighted_basis_value_dx_col = basis_value_dxi_col * weights * coef_alpha  
#             jac += weighted_basis_value_dx_col.t() @ basis_value_dxi_col 

    print("assembling the mass matrix time taken: ", time.time()-start_time) 

    start_time = time.time()    
    if solver == "cg": 
        sol, exit_code = linalg.cg(np.array(jac.detach().cpu()),np.array(rhs.detach().cpu()),tol=1e-12)
        sol = torch.tensor(sol).view(1,-1)
    elif solver == "direct": 
#         sol = np.linalg.inv( np.array(jac.detach().cpu()) )@np.array(rhs.detach().cpu())
        sol = (torch.linalg.solve( jac.detach(), rhs.detach())).view(1,-1)
    elif solver == "ls":
        sol = (torch.linalg.lstsq(jac.detach().cpu(),rhs.detach().cpu(),driver='gelsd').solution).view(1,-1)
        # sol = (torch.linalg.lstsq(jac.detach(),rhs.detach()).solution).view(1,-1) # gpu/cpu, driver = 'gels', cannot solve singular
    print("solving Ax = b time taken: ", time.time()-start_time)
    return sol 

In [8]:
d = 6  

def u_exact(x):
    z = torch.prod(torch.sin(pi/2 * x),dim = 1,keepdim = True)
    return z 

def target(x):
    z = (d * (pi/2)**2 + 1)*torch.prod(torch.sin(pi/2 * x),dim = 1,keepdim = True) 
    return z

def u_exact_grad():
    dim = d # d is global variable defined outside the function 
    grad_list = [] 
    def make_u_i(i):
        def u_i(x): 
            sin_terms = torch.sin(pi/2 * x)
            cos_term = torch.cos(pi/2 * x[:, i-1:i])
            sin_terms[:, i-1:i] = cos_term  # Replace the first sine term with the cosine term
            z = (pi/2) * torch.prod(sin_terms, dim=1, keepdim=True)
            return z 
        
        return u_i 
    for i in range(1,dim+1):
        grad_list.append(make_u_i(i))
    return grad_list


M = int(5e7) 
integration_weights, integration_points = MonteCarlo_Sobol_dDim_weights_points(M, d, -1, 1) 

neuron_num_list = [25,50,100,200,400,800,1600,3200]
actual_neuron_list = []
err_list_l2 = []
err_list_h1 = []
relu_k = 4  
for neuron_num in neuron_num_list: 
    my_model = model(d,neuron_num,1,relu_k).to(device)
    my_model = initialize_model_2(my_model,dim=d).to(device)
    my_model = remove_redundant_neuron(my_model.cpu(), dims = d, choice = 2).to(device) 
    solver = 'direct'
    # sol = minimize_linear_layer_H1_explicit_assemble_efficient(my_model,target,integration_weights, integration_points,activation = 'relu',solver = solver)
    sol = minimize_linear_layer_H1_explicit_assemble_efficient_general_dim(my_model,target,None,integration_weights, integration_points,torch.tensor([]),torch.tensor([]),activation = 'relu',solver = solver,memory = 2**27)
    my_model.fc2.weight.data[0,:] = sol[:] 
    actual_neuron_list.append(my_model.fc1.bias.size(0)) 
    # compute the error 
    # compute the error 
    memory = 2**28
    num_neuron = 0 if my_model == None else int(my_model.fc1.bias.detach().data.size(0))
    total_size2 = M*(num_neuron+1)
    num_batch2 = total_size2//memory + 1 
    batch_size_2 = M//num_batch2 #
    
    errl2 = compute_l2_error(u_exact,my_model,M,batch_size_2,integration_weights,integration_points)
#     errh1 = integration_weights.t()@(u_x(integration_points) - my_model.evaluate_derivative(integration_points,1).detach())**2
#     errh1 += integration_weights.t()@(u_y(integration_points) - my_model.evaluate_derivative(integration_points,2).detach())**2
#     errh1 += integration_weights.t()@(u_z(integration_points) - my_model.evaluate_derivative(integration_points,3).detach())**2 
#     errh1 = errh1**0.5    
    errh1 = compute_gradient_error(u_exact_grad,my_model,M,batch_size_2,integration_weights,integration_points)

    print("L2 error: ",errl2)
    print("H1 error: ",errh1) 
    err_list_l2.append(errl2.item())
    err_list_h1.append(errh1.item())
    # plot_2D(my_model.cpu())  

Number of neurons removed:  0
Number of neurons left:  25
total size: 25 50000000 = 1250000000
num batches:  10
assembling the mass matrix time taken:  0.01239013671875
solving Ax = b time taken:  0.6861240863800049
L2 error:  tensor(0.9953564850906, device='cuda:0')
H1 error:  tensor(3.8346094445063, device='cuda:0')
Number of neurons removed:  0
Number of neurons left:  50
total size: 50 50000000 = 2500000000
num batches:  19
assembling the mass matrix time taken:  0.49862146377563477
solving Ax = b time taken:  0.7840807437896729
L2 error:  tensor(0.9974356983647, device='cuda:0')
H1 error:  tensor(3.8407490569824, device='cuda:0')
Number of neurons removed:  0
Number of neurons left:  100
total size: 100 50000000 = 5000000000
num batches:  38
assembling the mass matrix time taken:  1.8509852886199951
solving Ax = b time taken:  0.9492805004119873
L2 error:  tensor(0.9980150646233, device='cuda:0')
H1 error:  tensor(3.8420054157047, device='cuda:0')
Number of neurons removed:  0
Num

In [9]:
d = 6 
print("L2 order:",0.5 + (2*relu_k +1)/(2*d))
print("H1 order:",0.5 + (2*(relu_k-1) +1)/(2*d))

output_convergence_order(actual_neuron_list,err_list_l2, err_list_h1)

L2 order: 1.25
H1 order: 1.0833333333333335
neuron num  & 	 $\|u-u_n \|_{L^2}$ & 	 order  & $ |u-u_n |_{H^1}$ & 	 order\\ \hline \hline 
25 		 & 9.954e-01 &		 * & 3.835e+00 &  *  \\ \hline  

50 		 &  9.974e-01 &  		 -0.00  &  3.841e+00 &  -0.00  \\ \hline  

100 		 &  9.980e-01 &  		 -0.00  &  3.842e+00 &  -0.00  \\ \hline  

200 		 &  9.848e-01 &  		 0.02  &  3.807e+00 &  0.01  \\ \hline  

399 		 &  9.036e-01 &  		 0.12  &  3.602e+00 &  0.08  \\ \hline  

799 		 &  5.389e-01 &  		 0.74  &  2.582e+00 &  0.48  \\ \hline  

1599 		 &  1.690e-01 &  		 1.67  &  1.132e+00 &  1.19  \\ \hline  

3191 		 &  6.037e-02 &  		 1.49  &  5.145e-01 &  1.14  \\ \hline  

